# Detección de smishing en el contexto peruano
**Hiroshi Alberto Takeda Chumpitazi** · Universidad de Lima

Este notebook arma el corpus de SMS peruanos, entrena y compara los modelos, y genera las tablas y figuras del artículo. Es el notebook v9 con las funciones separadas en el paquete `src/smishing_peru/` (un módulo por sección); los pasos y los resultados son los mismos.

**Cómo correrlo en Colab**
1. Ejecuta todas las celdas. La primera descarga el repositorio y prepara la carpeta `trabajo/`, donde quedan todas las salidas. La segunda instala las versiones de la corrida del artículo; si muestra un aviso, reinicia la sesión (Entorno de ejecución > Reiniciar sesión) y vuelve a ejecutar todo.
2. Cuando la sección 1 lo pida, sube los 3 CSV de referencia: `SMSSpamCollection.csv`, `Dataset_5971.csv` y `analysisdataset.csv` (de dónde bajarlos: `data/referencia/README.md`).
3. Los checkpoints de Gemini de la corrida del 25 de octubre (`data/checkpoints/`) se copian solos a `trabajo/`: no se llama a Gemini y el corpus sale igual.

La API key de Gemini solo hace falta para armar un corpus nuevo (`REGENERAR_CON_GEMINI = True` en 2.1). Va en Colab Secrets con el nombre `GEMINI_API_KEY`.

### Historial del notebook
**Qué cambia respecto de la v8:**
1. **Checkpoints de Gemini obligatorios (2.1):** si falta alguno, el notebook se detiene y avisa, en lugar de volver a llamar a Gemini y armar un corpus distinto sin decir nada. En la corrida de la v8 en Colab pasó eso y cambiaron todos los números.
2. **Prueba extra 5.6 (c):** el modelo final se reentrena con la mitad de IMC25 y se evalúa en la otra mitad y en el test propio. No cambia el modelo final ni ningún resultado anterior. Hoja nueva `11g_imc25_entrenando` en el Excel.

**Qué cambió de la v7 a la v8:** se agregó una prueba extra con smishing real en español (5.6 b, IMC25): mide qué porcentaje de esos fraudes detecta el modelo final, sin usarlos para entrenar. Necesita internet en Colab para descargar IMC25; si no hay, la celda avisa y sigue. Los nombres de los archivos de salida no cambian.

**Qué cambió de la v6 a la v7** (pedidos de la revisión de la exposición):
1. **Los SMS generados solo sirven para entrenar (5.0):** son *data augmentation*. El test sale solo de los SMS adaptados de los datasets reales (80/20, por grupos), y en la validación cruzada y el Grid Search los pliegues de validación también tienen solo SMS reales (`ValidacionSoloReales`). Ya no hay un test mixto.
2. **Enlaces del mensaje original (3.0):** Gemini inventaba enlaces peruanos, casi siempre con el nombre de la entidad en el dominio. Ahora cada SMS adaptado lleva el enlace de su original en inglés; se excluyen los SMS en los que Gemini agregó un enlace que el original no tenía. La versión para publicar desactiva esos enlaces reales (`hxxp`, `[.]`) en lugar de inventar otros (3.4). El preprocesamiento también reconoce enlaces ofuscados (`REGEX_ENLACE_AMPLIO`, 2.2).
3. **Aporte de los SMS generados (5.4 a y b):** el modelo final se compara con el mismo modelo entrenado solo con los SMS reales y solo con los generados, en el mismo test real. Se quitó la validación por escenarios no vistos, porque evaluaba sobre SMS generados.
4. **Sección 9:** tabla por pliegue (9.1), curvas de exactitud y pérdida del mejor modelo (9.2) y resultados con y sin spam (9.3), todo guardado en `experimentos_v7_TakedaHiroshi.zip` (9.4).

Los cambios de la v5 y la v6 están descritos en el encabezado de sus notebooks.

## 0. Repositorio y dependencias
La primera celda descarga el repositorio (en tu computadora, abre el notebook desde la carpeta `notebooks/` del repositorio) y prepara la carpeta `trabajo/`: ahí se leen los CSV y se guardan todas las salidas. La segunda instala las versiones de la corrida del artículo.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Al ordenar, numpy desempata distinto según el procesador: con AVX-512 o sin él. Eso cambia qué
# 3,000 términos entran al TF-IDF y, con ellos, los resultados. La corrida del artículo se hizo
# en un procesador sin AVX-512, y con esta línea numpy ordena igual en cualquiera. Tiene que ir
# antes de importar numpy; si numpy ya estaba cargado, la celda siguiente lo avisa.
os.environ["NPY_DISABLE_CPU_FEATURES"] = ("AVX512F AVX512CD AVX512_KNL AVX512_KNM AVX512_SKX AVX512_CLX "
                                          "AVX512_CNL AVX512_ICL AVX512_SPR")

REPO_URL = "https://github.com/DareekShot/smishing-peru.git"


def _buscar_raiz(carpeta):
    for c in [carpeta, *carpeta.parents]:
        if (c / "src" / "smishing_peru").is_dir():
            return c
    return None


INICIO = Path.cwd().resolve()
RAIZ = _buscar_raiz(INICIO)
if RAIZ is None:
    RAIZ = INICIO / "smishing-peru"
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(RAIZ)], check=True)

SRC = str(RAIZ / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
# Para que los procesos en paralelo de scikit-learn también encuentren el paquete
os.environ["PYTHONPATH"] = os.pathsep.join(p for p in [SRC, os.environ.get("PYTHONPATH")] if p)

from smishing_peru.carpeta_trabajo import preparar_carpeta_trabajo

os.chdir(preparar_carpeta_trabajo(RAIZ, buscar_tambien_en=[INICIO]))
print("Carpeta de trabajo:", Path.cwd())

In [2]:
# Las versiones de la corrida del artículo (requirements.txt). scikit-learn tiene que ser 1.6.1: la
# partición train/test (StratifiedGroupKFold con shuffle) cambia entre versiones.
!pip install -q -r "{RAIZ}/requirements.txt"

import warnings
warnings.filterwarnings("ignore")

import sys
import sklearn, xgboost, numpy, pandas, scipy
print(f"Python {sys.version.split()[0]} | scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__} | "
      f"numpy {numpy.__version__} | pandas {pandas.__version__} | scipy {scipy.__version__}")

ESPERADAS = {"scikit-learn": "1.6.1", "xgboost": "3.4.1", "numpy": "2.1.3", "pandas": "2.2.3", "scipy": "1.16.3"}
actuales = {"scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__, "numpy": numpy.__version__,
            "pandas": pandas.__version__, "scipy": scipy.__version__}
avisos = [f"{k} {actuales[k]} (la corrida usó {v})" for k, v in ESPERADAS.items() if actuales[k] != v]
try:
    from numpy._core._multiarray_umath import __cpu_features__
    if __cpu_features__.get("AVX512_SKX"):
        avisos.append("numpy se cargó antes de la primera celda y está usando AVX-512")
except ImportError:
    pass
if avisos:
    print("Ojo, los resultados pueden cambiar: " + "; ".join(avisos) + ". Reinicia la sesión "
          "(Entorno de ejecución > Reiniciar sesión) y vuelve a ejecutar todo.")

Python 3.13.16 | scikit-learn 1.6.1 | xgboost 3.4.1 | numpy 2.1.3 | pandas 2.2.3 | scipy 1.16.3


## 1. Integración de los datasets de referencia



In [ ]:
import os

ARCHIVOS_REQUERIDOS = ["SMSSpamCollection.csv", "Dataset_5971.csv", "analysisdataset.csv"]
faltantes = [f for f in ARCHIVOS_REQUERIDOS if not os.path.exists(f)]

if faltantes:
    try:
        from google.colab import files
        print(f"Faltan estos archivos en el entorno: {faltantes}")
        print("Sube ahora los 3 CSV del proyecto (puedes seleccionarlos todos juntos):")
        files.upload()
    except ImportError:
        raise FileNotFoundError(
            f"No se encontraron los archivos {faltantes} en el directorio actual. "
            "Colócalos en data/referencia/ (o en la carpeta trabajo/) antes de continuar."
        )
else:
    print("Los 3 datasets ya están disponibles en el entorno. Continuando...")

Faltan estos archivos en el entorno: ['SMSSpamCollection.csv', 'Dataset_5971.csv', 'analysisdataset.csv']
Sube ahora los 3 CSV del proyecto (puedes seleccionarlos todos juntos):


In [ ]:
# Las funciones y constantes de esta celda están en src/smishing_peru/datos_referencia.py.
import random

import numpy as np
import pandas as pd

from smishing_peru.datos_referencia import (
    cargar_analysis_dataset, cargar_dataset_5971, cargar_uci_sms,
)

random.seed(42)
np.random.seed(42)

# --- Integracion: un solo DataFrame de referencia con los 3 datasets normalizados ---
referencia = pd.concat([
    cargar_uci_sms("SMSSpamCollection.csv"),
    cargar_dataset_5971("Dataset_5971.csv"),
    cargar_analysis_dataset("analysisdataset.csv"),
], ignore_index=True)
referencia = referencia.dropna(subset=["label", "text"]).drop_duplicates(subset="text")


print("Dataset de referencia integrado:", referencia.shape)
print(referencia.groupby(["source", "label"]).size())
referencia = referencia.reset_index(drop=True)
print("\nEtiqueta original por fuente:")
print(referencia.groupby(["source", "label_original"]).size())

## 2. Transformación semántica y Data Augmentation con Gemini

### 2.1 Checkpoints de Gemini (y API key, solo para un corpus nuevo)

In [ ]:
import os

MODELO = "gemini-3.1-flash-lite"  # alternativa más potente: "gemini-3-flash-preview"

# Gemini responde distinto en cada llamada: si se vuelve a llamar, sale otro corpus y cambian
# todos los números. Por eso, por defecto, el corpus se arma solo con los checkpoints guardados
# (las respuestas de Gemini de la corrida del 25 de octubre) y NO se llama a Gemini. Si falta
# alguno, el notebook se detiene aquí en lugar de armar un corpus nuevo sin avisar.
# Pon REGENERAR_CON_GEMINI = True solo si quieres un corpus nuevo (todos los números cambiarán).
REGENERAR_CON_GEMINI = False
CHECKPOINTS_GEMINI = ["dataset_traducido_adaptado.csv", "corpus_gemini_peru_v2.csv"]

if not REGENERAR_CON_GEMINI:
    faltan = [f for f in CHECKPOINTS_GEMINI if not os.path.exists(f)]
    if faltan:
        raise FileNotFoundError(
            f"Faltan los checkpoints de Gemini: {', '.join(faltan)}. Súbelos a Colab con esos nombres "
            "(carpeta checkpoints_corrida_25oct) para que el corpus salga igual que en la corrida del "
            "25 de octubre. Si quieres un corpus nuevo, pon REGENERAR_CON_GEMINI = True.")
    client = None
    print("Se usan los checkpoints guardados: no se llama a Gemini y el corpus sale igual que en la corrida anterior.")
else:
    from google import genai

    def _obtener_api_key():
        # 1) Intenta Colab Secrets (recomendado): icono de llave en el panel izquierdo,
        #    crea un secreto llamado GEMINI_API_KEY y actívalo para este notebook.
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
            if key:
                print("API key cargada desde Colab Secrets.")
                return key
        except Exception:
            pass
        # 2) Variable de entorno (útil si corres esto fuera de Colab)
        if os.environ.get("GEMINI_API_KEY"):
            print("API key cargada desde variable de entorno.")
            return os.environ["GEMINI_API_KEY"]
        # 3) Último recurso: pedirla de forma oculta (no queda en texto plano en el notebook)
        import getpass
        print("No se encontró la key en Secrets ni en el entorno.")
        return getpass.getpass("Pega tu GEMINI_API_KEY (no se mostrará en pantalla): ")

    API_KEY = _obtener_api_key()
    client = genai.Client(api_key=API_KEY)
    print("Cliente de Gemini listo. Ojo: el corpus que salga será distinto al de la corrida anterior.")

### 2.2 Catálogo peruano de entidades y escenarios



In [ ]:
# Catálogo peruano: entidades por sector, escenarios de fraude y escenarios legítimos.
# Es la "fuente única de verdad" del contexto peruano: la usan la adaptación del
# corpus internacional (2.4), la generación ampliada (2.5), el extractor de
# características (4) y el diagnóstico explicativo (6).
# El código de esta celda está en src/smishing_peru/catalogo.py.
from smishing_peru.catalogo import (
    CANALES_SMISHING, CATALOGO_ENTIDADES, ESCENARIOS_HAM, ESCENARIOS_SMISHING, ESTILOS,
    ESTILOS_HAM, REGEX_ENLACE_AMPLIO, SECTOR_DE_ENTIDAD, normalizar,
)

print(f"Catálogo: {len(SECTOR_DE_ENTIDAD)} entidades en {len(CATALOGO_ENTIDADES)} sectores")
print(f"Escenarios de smishing: {sum(len(v) for v in ESCENARIOS_SMISHING.values())}  |  "
      f"escenarios legítimos: {sum(len(v) for v in ESCENARIOS_HAM.values())}")
print(f"Canales de ataque: {list(CANALES_SMISHING)}  |  estilos smishing: {list(ESTILOS)}  |  estilos HAM: {list(ESTILOS_HAM)}")

### 2.3 Utilidades de llamada a Gemini (salida JSON con esquema, reintentos, detección de rechazos)

El código de esta celda está en [`src/smishing_peru/gemini.py`](../src/smishing_peru/gemini.py).

### 2.4 Traducción y adaptación semántica del corpus internacional



In [ ]:
# Traducción y adaptación semántica del corpus internacional al contexto peruano.
# No es una traducción literal: Gemini conserva la intención y el mecanismo
# persuasivo de cada SMS real, y reemplaza marcas, montos y lugares extranjeros
# por equivalentes peruanos del catálogo (criterio de inclusión de la ficha).
# El código de esta celda está en src/smishing_peru/adaptacion.py.
from smishing_peru.adaptacion import adaptar_corpus_internacional, seleccionar_candidatos

In [ ]:
# Ejecutar la traducción y adaptación (según la ficha: 2,000 SMS = 1,000 por clase).
# Se adapta ~10% de más para compensar lo que elimine el control de calidad (sección 3).
# Si se corta (cuota de la API, desconexión de Colab), vuelve a correr esta celda:
# retoma desde 'dataset_traducido_adaptado.csv' sin repetir lo ya hecho.
N_ADAPTAR_POR_CLASE = 1000
MARGEN_ADAPTACION = 1.10
TAM_LOTE_ADAPTACION = 15

if client is None:
    # Sin Gemini, el checkpoint tiene que estar completo (si no, la función intentaría llamarlo).
    _ckpt = pd.read_csv("dataset_traducido_adaptado.csv")
    if ((_ckpt["label"] == "HAM") & ~_ckpt["descartado"].astype(bool)).sum() < int(N_ADAPTAR_POR_CLASE * MARGEN_ADAPTACION):
        raise RuntimeError("El checkpoint 'dataset_traducido_adaptado.csv' está incompleto: súbelo completo "
                           "o pon REGENERAR_CON_GEMINI = True en 2.1.")

df_adaptacion = adaptar_corpus_internacional(
    client, MODELO, referencia,
    n_por_clase=int(N_ADAPTAR_POR_CLASE * MARGEN_ADAPTACION),
    tam_lote=TAM_LOTE_ADAPTACION,
    archivo_checkpoint="dataset_traducido_adaptado.csv",
    pausa_entre_llamadas=1.0,
    # Si ya tienes el checkpoint completo, no se vuelve a llamar a Gemini (el corpus sale idéntico).
    # Ponlo en False solo si quieres que Gemini intente adaptar los candidatos que faltan.
    solo_checkpoint=os.path.exists("dataset_traducido_adaptado.csv"),
)

df_adaptado = df_adaptacion[~df_adaptacion["descartado"].astype(bool)].copy()
df_adaptado["source"] = "traduccion_adaptada"
df_adaptado["origen"] = "adaptado_internacional"

print("\n=== Resumen de la adaptación ===")
print("SMS procesados:", len(df_adaptacion))
print("Adaptados por clase:\n", df_adaptado["label"].value_counts().to_string())
descartados = df_adaptacion[df_adaptacion["descartado"].astype(bool)]
print(f"\nDescartados por criterio de exclusión (spam publicitario sin fraude u otros): {len(descartados)}")
print(descartados.groupby(["source_original", "label"]).size().to_string() if len(descartados) else "(ninguno)")
print("\nGatillo asignado a los SMISHING adaptados:\n",
      df_adaptado.loc[df_adaptado["label"] == "SMISHING", "trigger"].value_counts(dropna=False).to_string())

# Ejemplos lado a lado: original en inglés -> versión peruana
pd.set_option("display.max_colwidth", 120)
pd.concat([g.sample(min(4, len(g)), random_state=3) for _, g in df_adaptado.groupby("label")])[
    ["label", "text_original", "text"]]

### 2.5 Generación ampliada y controlada



In [ ]:
# Generación ampliada con Gemini: en vez de pedir "180 mensajes de urgencia" y confiar
# en que el modelo varíe solo, cada llamada recibe una combinación distinta de
# gatillo x escenario x entidad x canal x estilo. La variedad queda garantizada por
# el plan (y es medible), no por la suerte.
# El código de esta celda está en src/smishing_peru/generacion.py.
from smishing_peru.generacion import generar_corpus_peru, planificar_generacion

In [ ]:
# Ejecutar la generación ampliada (según la ficha: 3,000 SMS = 1,500 por clase).
# Se pide ~30% de más: compensa lo que elimina el control de calidad (duplicados y
# casi-duplicados) y deja reserva para completar la clase SMISHING, porque la adaptación
# solo encontró ~760 SMS fraudulentos reales válidos (no 1,000). Con 1.12 el dataset
# quedaba en 4,940 SMS. Retomable: si ya tienes 'corpus_gemini_peru_v2.csv', solo se
# hacen las llamadas nuevas del plan ampliado y del refuerzo.
N_GENERAR_POR_CLASE = 1500
MARGEN_CONTROL_CALIDAD = 1.30
TAM_LOTE_GENERACION = 10

plan_generacion = planificar_generacion(
    n_smishing=int(N_GENERAR_POR_CLASE * MARGEN_CONTROL_CALIDAD),
    n_ham=int(N_GENERAR_POR_CLASE * MARGEN_CONTROL_CALIDAD),
    tam_lote=TAM_LOTE_GENERACION,
)
print("Llamadas planificadas por clase y categoría:")
print(plan_generacion.groupby(["label", "categoria"]).size().to_string())
print("\nCanal de ataque en el plan de SMISHING:")
print(plan_generacion[plan_generacion["label"] == "SMISHING"]["canal"].value_counts().to_string())

if client is None:
    # Sin Gemini, el checkpoint tiene que cubrir todas las llamadas del plan.
    _faltan = (~plan_generacion["plan_id"].isin(set(pd.read_csv("corpus_gemini_peru_v2.csv")["plan_id"]))).sum()
    if _faltan:
        raise RuntimeError(f"Al checkpoint 'corpus_gemini_peru_v2.csv' le faltan {_faltan} llamadas del plan: "
                           "súbelo completo o pon REGENERAR_CON_GEMINI = True en 2.1.")

df_generado = generar_corpus_peru(client, MODELO, plan_generacion,
                                  archivo_checkpoint="corpus_gemini_peru_v2.csv",
                                  pausa_entre_llamadas=1.0)

print("\nRefuerzo incluido (familiar que cambió de número / avisos legítimos con teléfono):")
print(df_generado[df_generado["categoria"].astype(str).str.startswith("refuerzo")]
      .groupby(["label", "categoria"]).size().to_string())
print("\n=== Resumen de la generación ===")
print(df_generado.groupby(["label", "categoria"]).size().to_string())
df_generado.sample(10, random_state=1)[["label", "categoria", "canal", "text"]]

## 3. Construcción del dataset final y control de calidad


In [ ]:
# Construcción del dataset final (en español peruano) con control de calidad.
# Las funciones y constantes de esta celda están en src/smishing_peru/corpus.py.
from smishing_peru.corpus import construir_dataset_final

df, informe_calidad = construir_dataset_final(
    df_adaptado, df_generado,
    n_adaptado_por_clase=N_ADAPTAR_POR_CLASE,
    n_generado_por_clase=N_GENERAR_POR_CLASE,
)

print("=== Control de calidad ===")
for paso, valor in informe_calidad.items():
    print(f"  {paso}: {valor}")
print("\nDataset final:", df.shape, "| por clase:", df["label"].value_counts().to_dict())
print(df.groupby(["origen", "label"]).size().to_string())
print("\nRefuerzo dentro del dataset final:")
print(df[df["categoria"].astype(str).str.startswith("refuerzo")].groupby(["label", "categoria"]).size().to_string())
if len(df) != 2 * (N_ADAPTAR_POR_CLASE + N_GENERAR_POR_CLASE):
    print("\nAVISO: el dataset no llegó a 5,000 SMS. Sube MARGEN_CONTROL_CALIDAD (p. ej. a 1.40) y vuelve "
          "a correr la celda de 2.5: solo hará las llamadas nuevas del plan ampliado.")
df.sample(8, random_state=7)[["label", "origen", "text"]]

### 3.0 Enlaces del mensaje original

En los SMS adaptados de datasets reales no se inventan enlaces: cada uno lleva el enlace de su mensaje original en inglés. Los SMS generados con Gemini no se tocan, porque solo se usan para entrenar.

In [ ]:
# --- 3.0 Enlaces: en los SMS adaptados se usa el enlace del mensaje original ---
# Las instrucciones de adaptación (2.4) le pedían a Gemini inventar un enlace peruano cuando
# el original traía uno, y Gemini solía poner el nombre de la entidad en el dominio (por
# ejemplo, bcp-seguro.pe). En los originales en inglés eso no es una regla: solo cerca de 1 de
# cada 4 SMISHING con enlace lleva en el dominio una palabra del mensaje. Para no inventar
# enlaces en los SMS que vienen de mensajes reales, a cada SMS adaptado se le pone el enlace
# de su original:
#  - si los dos tienen enlace, los del adaptado se reemplazan por los del original, en orden;
#  - si el original tenía enlace y el adaptado no, se deja como está (Gemini cambió el canal,
#    por ejemplo a un teléfono);
#  - si el original NO tenía enlace y Gemini agregó uno, el SMS se excluye: quitar el enlace
#    dejaría la frase incompleta ("ingrese aquí para ...").
# Los SMS generados no se tocan: son sintéticos y solo se usan para entrenar (sección 5.0).
# El modelo no lee el dominio, porque los enlaces pasan a "urltoken" antes del TF-IDF.
# Las funciones y constantes de esta celda están en src/smishing_peru/corpus.py.
from smishing_peru.corpus import usar_enlace_original

acciones = []
for i in df.index[df["origen"] == "adaptado_internacional"]:
    df.at[i, "text"], accion = usar_enlace_original(df.at[i, "text"], df.at[i, "text_original"])
    acciones.append({"i": i, "label": df.at[i, "label"], "Qué se hizo con el enlace": accion})
acciones = pd.DataFrame(acciones)
tabla_enlaces = pd.crosstab(acciones["Qué se hizo con el enlace"], acciones["label"], margins=True, margins_name="Total")
print("SMS adaptados: enlace del mensaje original en lugar del que puso Gemini")
print(tabla_enlaces.to_string())

excluir = acciones.loc[acciones["Qué se hizo con el enlace"].str.endswith("(se excluye)"), "i"]
df = df.drop(index=excluir).reset_index(drop=True)
print(f"\nDataset final: {len(df)} SMS ({len(excluir)} excluidos) | por clase: {df['label'].value_counts().to_dict()}")
print(df.groupby(["origen", "label"]).size().to_string())
print(f"SMS duplicados después del cambio: {df['text'].duplicated().sum()} (debe ser 0)")

### 3.1 Vista general del dataset integrado

In [ ]:
# --- Vista general: filas, columnas y tipos de datos ---
print("Forma del dataset final (filas, columnas):", df.shape)
print()
df.info()

In [ ]:
# --- Tabla: primeras filas del dataset (para verlo como tabla en Colab) ---
# Solo las columnas del modelo. El texto original en inglés de los SMS adaptados queda en
# la columna 'text_original' (trazabilidad), pero el modelo nunca la usa.
df[["text", "label", "trigger", "origen", "source_original", "entidades"]].head(15)

In [ ]:
# --- Muestra aleatoria (para ver ejemplos variados, no solo los primeros) ---
df[["text", "label", "trigger", "origen", "source_original", "entidades"]].sample(15, random_state=1)

In [ ]:
# --- Conteo de clases y de dónde viene cada mensaje ---
print("Distribución de clases (HAM vs SMISHING):")
print(df["label"].value_counts())
print()
print("Distribución por fuente y clase:")
print(df.groupby(["source", "label"]).size())

In [ ]:
# --- Gráfico: balance de clases y composición por fuente ---
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.countplot(data=df, x="label", hue="label", palette="Blues_d", legend=False, ax=axes[0])
axes[0].set_title("Balance de clases: HAM vs SMISHING")
axes[0].set_xlabel("")
axes[0].set_ylabel("Cantidad de mensajes")

conteo_fuente = df.groupby(["source", "label"]).size().unstack(fill_value=0)
conteo_fuente.plot(kind="barh", stacked=True, ax=axes[1], color=["#4C72B0", "#DD8452"])
axes[1].set_title("Composición del dataset por fuente")
axes[1].set_xlabel("Cantidad de mensajes")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

### 3.2 Diversidad del dataset y chequeo de atajos


In [ ]:
# Informe de diversidad y chequeo de "atajos" (shortcuts).
# Un atajo es una señal trivial que separa las clases sin que el modelo entienda el
# fraude (p. ej. "todo SMISHING tiene URL y ningún HAM la tiene"). Si fuera muy fuerte,
# las métricas saldrían infladas y no se sostendrían con SMS reales. Aquí se MIDE.
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

aux = pd.DataFrame({
    "label": df["label"], "origen": df["origen"], "trigger": df["trigger"],
    "entidades": df["entidades"].fillna("").map(lambda s: [e for e in str(s).split("|") if e]),
    "tiene_url": df["text"].map(lambda t: bool(REGEX_ENLACE_AMPLIO.search(t))),
    "largo": df["text"].str.len(),
})
aux["tiene_entidad"] = aux["entidades"].map(bool)
aux["sectores"] = aux["entidades"].map(lambda ents: sorted({SECTOR_DE_ENTIDAD[e] for e in ents}))

entidades_usadas = aux.explode("entidades")["entidades"].dropna()
print(f"Entidades distintas presentes en el dataset: {entidades_usadas.nunique()} de {len(SECTOR_DE_ENTIDAD)} del catálogo")
print(f"Sectores cubiertos: {aux.explode('sectores')['sectores'].dropna().nunique()} de {len(CATALOGO_ENTIDADES)}")

print("\n--- Señales simples por origen y clase ---")
print(aux.groupby(["origen", "label"]).agg(con_url=("tiene_url", "mean"),
                                          menciona_entidad=("tiene_entidad", "mean"),
                                          largo_promedio=("largo", "mean")).round(3).to_string())

# Línea base de "un solo atajo": ¿qué F1 logra un clasificador trivial que mira UNA señal?
# El modelo completo tiene que superar claramente estas cifras para que su F1 signifique algo.
y_bin = (aux["label"] == "SMISHING").astype(int)
cv_simple = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lineas_base = {
    "Solo '¿tiene URL?'": f1_score(y_bin, aux["tiene_url"].astype(int)),
    "Solo longitud (árbol de 1 corte)": f1_score(y_bin, cross_val_predict(
        DecisionTreeClassifier(max_depth=1, random_state=42), aux[["largo"]], y_bin, cv=cv_simple)),
    "URL + longitud (árbol de 2 niveles)": f1_score(y_bin, cross_val_predict(
        DecisionTreeClassifier(max_depth=2, random_state=42), aux[["largo", "tiene_url"]].astype(float), y_bin, cv=cv_simple)),
}
print("\n--- F1 de clasificadores triviales (línea base de atajos) ---")
for nombre, f1 in lineas_base.items():
    print(f"  {nombre}: F1 = {f1:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
top = entidades_usadas.value_counts().head(20)
sns.barplot(x=top.values, y=top.index, ax=axes[0], color="#4C72B0")
axes[0].set_title("Top 20 entidades mencionadas")
axes[0].set_xlabel("Mensajes")

sectores = aux.explode("sectores").dropna(subset=["sectores"]).reset_index(drop=True)
pd.crosstab(sectores["sectores"], sectores["label"]).plot(kind="barh", ax=axes[1], color=["#55A868", "#C44E52"])
axes[1].set_title("Sector de la entidad mencionada, por clase")
axes[1].set_xlabel("Mensajes")
axes[1].set_ylabel("")

smi = aux[aux["label"] == "SMISHING"]
pd.crosstab(smi["trigger"].fillna("sin asignar"), smi["origen"]).plot(kind="barh", ax=axes[2],
                                                                       color=["#8172B2", "#DD8452"])
axes[2].set_title("Gatillo psicológico en SMISHING, por origen")
axes[2].set_xlabel("Mensajes")
axes[2].set_ylabel("")
plt.tight_layout()
plt.show()

### 3.3 Trazabilidad de las entradas

Esta sección muestra qué datos entraron y de dónde vienen: el recorrido de cada fuente desde los CSV originales hasta los 5,000 SMS finales. Cada etapa se guarda en la carpeta `evidencias/`.

In [ ]:
# --- 3.3 Trazabilidad: el recorrido de los datos, desde los 3 CSV originales hasta el dataset final ---
# Cada etapa muestra cuántos SMS hay de cada fuente y clase, y todo se guarda en la
# carpeta "evidencias/" para poder revisarlo fuera del notebook.
# Las funciones y constantes de esta celda están en src/smishing_peru/corpus.py.
from smishing_peru.corpus import NOMBRE_FUENTE, _conteo_por_fuente, fuente_detallada

os.makedirs("evidencias", exist_ok=True)

candidatos_ref = seleccionar_candidatos(referencia)  # el mismo filtro que usa la sección 2.4
procesados = df_adaptacion.assign(descartado=df_adaptacion["descartado"].astype(bool))
generados = df_generado.assign(fuente=np.where(df_generado["categoria"].astype(str).str.startswith("refuerzo"),
                                               "gemini_refuerzo", "gemini_llm_peru"))

tabla_embudo = pd.concat([
    _conteo_por_fuente(referencia, "1. CSV originales en inglés (unificados, sin duplicados exactos)", "source"),
    _conteo_por_fuente(pd.concat([candidatos_ref["SMISHING"], candidatos_ref["HAM"]]),
                       "2. Candidatos a adaptar (20-400 caracteres, sin casi-duplicados)", "source"),
    _conteo_por_fuente(procesados[~procesados["descartado"]], "3a. Traducidos y adaptados al Perú por Gemini", "source_original"),
    _conteo_por_fuente(procesados[procesados["descartado"]], "3b. Descartados (publicidad sin fraude u otros)", "source_original"),
    _conteo_por_fuente(generados, "4. Generados con Gemini (catálogo peruano)", "fuente"),
    _conteo_por_fuente(df.assign(fuente=fuente_detallada(df)), "5. Dataset final (control de calidad, muestreo y enlace del original)", "fuente"),
], ignore_index=True)

print("Etapa 2: el HAM solo se toma de UCI y Dataset_5971 (en analysisdataset, 'HAM' solo significa")
print("'URL no detectada', no 'legítimo verificado'). Hay más HAM candidato del necesario, así que")
print("se adaptó en orden aleatorio hasta llegar a 1,100. El SMISHING se adaptó completo.\n")
print("Control de calidad del paso 4 al 5:")
for paso, valor in informe_calidad.items():
    print(f"  {paso}: {valor}")

# Todo lo que entra y sale de cada etapa, guardado como CSV
referencia.to_csv("evidencias/01_referencia_original_integrada.csv", index=False)
df_adaptacion.to_csv("evidencias/02_traduccion_adaptacion_original_vs_peruano.csv", index=False)
df_generado.to_csv("evidencias/03_generado_gemini.csv", index=False)
df.assign(fuente=fuente_detallada(df)).to_csv("evidencias/04_dataset_final.csv", index=False)
tabla_embudo.to_csv("evidencias/00_recorrido_de_los_datos.csv", index=False)
print("\nGuardado en evidencias/: 00_recorrido_de_los_datos, 01_referencia_original_integrada, "
      "02_traduccion_adaptacion_original_vs_peruano, 03_generado_gemini, 04_dataset_final")
tabla_embudo

In [ ]:
# --- 3.4 Versión del corpus para publicar: enlaces reales desactivados ---
# Desde la v7, los SMS adaptados llevan el enlace real de su mensaje original (3.0), y varios
# son enlaces maliciosos reales (por ejemplo, los de Smishtank). Para publicar el corpus se
# desactivan con la convención habitual en seguridad: "http" pasa a "hxxp" y los puntos del
# dominio a "[.]", así nadie los abre por accidente. No se inventa ningún enlace, y el dataset
# de modelado no cambia.
# Las funciones y constantes de esta celda están en src/smishing_peru/corpus.py.
from smishing_peru.corpus import desactivar_enlaces

df_publico = df.drop(columns=["text_original"]).copy()
es_adaptado = df_publico["origen"] == "adaptado_internacional"
df_publico.loc[es_adaptado, "text"] = df_publico.loc[es_adaptado, "text"].map(desactivar_enlaces)
print(f"SMS adaptados con enlaces desactivados en la versión para publicar: "
      f"{(df_publico['text'] != df['text']).sum()}")
print("Ejemplo:", df_publico.loc[df_publico["text"] != df["text"], "text"].iloc[0])
df_publico.to_csv("evidencias/04b_dataset_publico.csv", index=False)
print("Guardado: evidencias/04b_dataset_publico.csv (para publicar: sin textos originales y con los "
      "enlaces reales desactivados)")

In [ ]:
# --- Composición final: de dónde viene cada SMS del dataset final ---
tabla_composicion = pd.crosstab(fuente_detallada(df).map(lambda f: NOMBRE_FUENTE.get(f, f)), df["label"], rownames=["Fuente"], colnames=["Clase"],
                                margins=True, margins_name="Total")
tabla_composicion["% del dataset"] = (tabla_composicion["Total"] / len(df) * 100).round(1)
tabla_composicion

In [ ]:
# --- Descargar los datasets como CSV (para revisarlos o adjuntarlos al informe) ---
df.to_csv("dataset_final_integrado_v4.csv", index=False)
try:
    from google.colab import files
    files.download("dataset_final_integrado_v4.csv")
    files.download("dataset_traducido_adaptado.csv")   # original en inglés + versión peruana
except ImportError:
    print("Archivos guardados: 'dataset_final_integrado_v4.csv' y 'dataset_traducido_adaptado.csv'.")

## 4. Preprocesamiento NLP y Extracción de Características


In [ ]:
# Preprocesamiento NLP y extractor de características estilísticas/estructurales
# v5: se agregan desofuscar() (defensa adversaria, opcional) y PreprocesadorTexto (para el pipeline).
# El código de esta celda está en src/smishing_peru/preprocesamiento.py y src/smishing_peru/caracteristicas.py.
from smishing_peru.caracteristicas import StylisticFeatureExtractor
from smishing_peru.preprocesamiento import (
    REGEX_CELULAR, PreprocesadorTexto, desofuscar, preprocesar_texto,
)

In [ ]:
# Demostración rápida del preprocesamiento y del extractor (con lógica difusa activada)
ejemplos = [
    "BCP: Su cuenta ha sido BLOQUEADA. Verifique urgente en bcp-verifica.info/x8k2",
    "Oe causa, ya te yapeé los S/ 25 de la pichanga, confirma pe",
    "Interbank: Realizaste un consumo de S/ 45.90 en Tottus con tu tarjeta ****1234.",
]

print("desofuscar() (solo se usa en la variante robusta de la sección 5.5):")
for e in ["BCP: su cuenta fue b l o q u e a d a, v3rifique en bcp-segur0.info", "ВСР: Su сuеntа fue bloqueada"]:
    print(f"  {e!r} -> {desofuscar(e)!r}")

extractor_demo = StylisticFeatureExtractor(usar_logica_difusa=True)
demo_df = pd.DataFrame(extractor_demo.transform(ejemplos), columns=extractor_demo.get_feature_names_out())
demo_df.insert(0, "texto", ejemplos)
demo_df.insert(1, "texto_preprocesado", [preprocesar_texto(e) for e in ejemplos])
demo_df

### 4.1 Dataset preprocesado

Esta sección muestra cómo queda cada SMS después del preprocesamiento y los valores de las 7 variables estilísticas. Se guarda completo.

In [ ]:
# --- 4.1 Dataset preprocesado (se guarda completo) ---
# Para todos los SMS se guarda el texto ya normalizado y tokenizado (quitando y sin quitar
# stopwords) y las 7 variables estilísticas crudas. Estas transformaciones son reglas fijas
# (no aprenden de los datos), así que calcularlas sobre todo el dataset no filtra
# información del test. El TF-IDF sí aprende (vocabulario e IDF): por eso se ajusta solo
# con train en la sección 5, y su vocabulario se guarda en la sección 8.
extractor_crudo = StylisticFeatureExtractor(usar_logica_difusa=False)
variables = pd.DataFrame(extractor_crudo.transform(df["text"].values),
                         columns=extractor_crudo.get_feature_names_out())
df_preprocesado = pd.concat([
    df[["text", "label", "origen"]].reset_index(drop=True).assign(fuente=fuente_detallada(df).values),
    pd.DataFrame({
        "preprocesado_quitando_stopwords": [preprocesar_texto(t, True) for t in df["text"]],
        "preprocesado_sin_quitar_stopwords": [preprocesar_texto(t, False) for t in df["text"]],
    }),
    variables,
], axis=1)
df_preprocesado.to_csv("evidencias/05_dataset_preprocesado.csv", index=False)

n_tokens = df_preprocesado["preprocesado_sin_quitar_stopwords"].str.split().str.len()
n_tokens_sin = df_preprocesado["preprocesado_quitando_stopwords"].str.split().str.len()
print(f"Tokens promedio por SMS: {n_tokens.mean():.1f} sin quitar stopwords -> {n_tokens_sin.mean():.1f} quitándolas")
print("\nPromedio de las 7 variables estilísticas por clase (así se ve qué separa a las clases):")
print(df_preprocesado.groupby("label")[list(variables.columns)].mean().round(3).T.to_string())
print("\nGuardado: evidencias/05_dataset_preprocesado.csv")
df_preprocesado.sample(10, random_state=3)

## 5. Entrenamiento y Comparación de Modelos de ML



In [ ]:
# Suite de entrenamiento y comparación de modelos de clasificación
# v5: cada modelo es un PIPELINE completo (SMS en texto plano -> predicción). Así, en la
# validación cruzada el TF-IDF y el escalado se reajustan dentro de cada pliegue, sin ver
# los SMS de validación (en la v4 se ajustaban una vez con todo train y luego se hacía la
# validación cruzada sobre esas matrices: una fuga leve que inflaba un poco el F1 de CV).
# v7: los SMS generados con Gemini (data augmentation) se usan solo para entrenar: el test y
# los pliegues de validación tienen solo SMS adaptados de datasets reales.
# El código de esta celda está en src/smishing_peru/modelos.py.
from smishing_peru.modelos import (
    RANDOM_STATE, ValidacionSoloReales, crear_pipeline, entrenar_y_comparar, fijar_parametro,
    medir_latencia_y_tamano, modelos_base, particion_por_grupos,
)

In [ ]:
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:0.4f}")

resultados, ctx = entrenar_y_comparar(df, usar_logica_difusa=True, quitar_stopwords=True)
print("Configuración: variables difusas, quitando stopwords")
print(f"Mejor modelo según F1 de validación cruzada: {resultados.iloc[0]['Modelo']}\n")
resultados

### 5.0 Partición train/test

Los SMS generados con Gemini son *data augmentation*: se usan solo para entrenar. El test sale únicamente de los SMS adaptados de los datasets reales (80/20, por grupos), y en la validación cruzada los pliegues de validación también tienen solo SMS reales. Esta tabla muestra qué SMS de cada fuente quedaron en train y en test. El test solo se usa al final, para reportar los resultados.

In [ ]:
# --- 5.0 Partición train/test: qué SMS hay en cada una ---
# La partición es la misma para todas las configuraciones y modelos (misma semilla y mismo
# dataset). Los SMS adaptados de datasets reales se dividen 80/20, estratificados por clase y
# dataset de origen, y por grupos (cada SMS adaptado es su propio grupo). Los SMS generados con
# Gemini son data augmentation: van todos a train y nunca se usan para evaluar. En la
# validación cruzada pasa lo mismo: los pliegues de validación tienen solo SMS reales y los
# generados se suman siempre a la parte de entrenamiento (ValidacionSoloReales, sección 5).
particion = np.array(["train"] * len(df), dtype=object)
particion[ctx["idx_test"]] = "test"
df_particion = df.assign(fuente=fuente_detallada(df), particion=particion)

grupos_train = set(df_particion.loc[df_particion["particion"] == "train", "grupo"])
grupos_test = set(df_particion.loc[df_particion["particion"] == "test", "grupo"])
print(f"Train: {len(ctx['idx_train'])} SMS ({len(ctx['idx_train']) / len(df):.1%})  |  "
      f"Test: {len(ctx['idx_test'])} SMS ({len(ctx['idx_test']) / len(df):.1%})")
print(f"Grupos que aparecen en train y en test a la vez: {len(grupos_train & grupos_test)} (debe ser 0)")
print(f"Test: solo SMS adaptados de datasets reales ({(df['origen'].values[ctx['idx_test']] == 'adaptado_internacional').mean():.0%}). "
      f"Los {(df['origen'] == 'generado_llm').sum()} SMS generados están todos en train.")
print("Validación cruzada y Grid Search: 5 y 3 pliegues dentro de train, por grupos, validando solo con SMS reales.\n")

df_particion[["text", "label", "trigger", "origen", "fuente", "grupo", "particion"]].to_csv(
    "evidencias/06_particion_train_test.csv", index=False)
print("Guardado: evidencias/06_particion_train_test.csv (cada SMS con su partición)")

tabla_particion = pd.crosstab([df_particion["origen"], df_particion["fuente"].map(lambda f: NOMBRE_FUENTE.get(f, f))],
                              [df_particion["particion"], df_particion["label"]], margins=True, margins_name="Total")
tabla_particion

### 5.1 Ablaciones: lógica difusa y stopwords

Se prueban las 4 combinaciones de dos decisiones del preprocesamiento: variables estructurales **difusas** (17 columnas) o **crudas** (7 columnas), y **quitando** o **sin quitar** las stopwords adaptadas al español peruano. En la v4, la configuración "sin stopwords" significaba en realidad *sin quitar* las stopwords (es decir, conservándolas); en la v5 el nombre lo dice explícitamente.

In [ ]:
# --- Ablaciones (diseño factorial 2 x 2). Criterio de decisión: F1 de validación cruzada
# (solo train). El test no participa en ninguna decisión: se usa una sola vez, al final.
COL_CV = "F1 CV (5-fold, media)"

configuraciones = {"Variables difusas, quitando stopwords": (resultados, ctx)}  # ya calculada en 5.0
for nombre, difusa, quitar in [("Variables crudas, quitando stopwords", False, True),
                               ("Variables difusas, sin quitar stopwords", True, False),
                               ("Variables crudas, sin quitar stopwords", False, False)]:
    print(f"Entrenando: {nombre}...")
    configuraciones[nombre] = entrenar_y_comparar(df, usar_logica_difusa=difusa, quitar_stopwords=quitar)

resumen_ablacion = pd.DataFrame([
    {"Configuración": nombre, "Mejor modelo (por F1 CV)": res.iloc[0]["Modelo"],
     "F1 CV": res.iloc[0][COL_CV], "F1 CV desv.": res.iloc[0]["F1 CV (desv. estandar)"]}
    for nombre, (res, _) in configuraciones.items()
]).sort_values("F1 CV", ascending=False).reset_index(drop=True)

# Gana el mayor F1 CV (4 decimales). En empate gana la más simple: este orden va de la más
# simple (7 variables, sin el paso de stopwords) a la más compleja.
orden_simplicidad = ["Variables crudas, sin quitar stopwords", "Variables crudas, quitando stopwords",
                     "Variables difusas, sin quitar stopwords", "Variables difusas, quitando stopwords"]
config_ganadora = max(orden_simplicidad, key=lambda k: round(configuraciones[k][0][COL_CV].max(), 4))
resultados_ganador, ctx_ganador = configuraciones[config_ganadora]

mejor_cv = resumen_ablacion["F1 CV"].iloc[0]
desv_cv = resumen_ablacion["F1 CV desv."].iloc[0]
rango = mejor_cv - resumen_ablacion["F1 CV"].min()
print(f"\nConfiguración ganadora: {config_ganadora}")
print(f"Diferencia entre la mejor y la peor configuración: {rango:.4f} "
      f"(desviación estándar del F1 CV de la mejor: {desv_cv:.4f})")
if rango < desv_cv:
    print("Las 4 configuraciones quedan a menos de una desviación estándar: ni la lógica difusa ni "
          "quitar las stopwords cambian el resultado de forma medible.")
resumen_ablacion

### 5.2 Ajuste de hiperparámetros (Grid Search) y ensambles (Voting + Stacking)



In [ ]:
# Ajuste de hiperparámetros (Grid Search) de los 4 modelos de la ficha y ensambles (Voting + Stacking)
# v5: la búsqueda se hace sobre el pipeline completo (el TF-IDF se reajusta en cada pliegue), y
# la calibración y el Stacking usan pliegues internos por grupos (CVPorGrupos).
# El código de esta celda está en src/smishing_peru/modelos.py.
from smishing_peru.modelos import afinar_hiperparametros_y_ensamblar

In [ ]:
resultados_afinados, modelos_afinados = afinar_hiperparametros_y_ensamblar(ctx_ganador)
resultados_afinados

### 5.3 Selección final del mejor modelo

Los candidatos se ordenan por F1 de validación cruzada (solo train, validando con SMS reales). Como el objetivo es un modelo ligero que corra en el celular, se elige el de mayor F1 entre los que pesan como máximo 1 MB y clasifican un SMS en 10 ms o menos (`SOLO_LIGEROS = True`). Recién después se reporta su resultado en el test.

In [ ]:
# --- Selección final: suite base (configuración ganadora) vs. afinados/ensambles ---
# Se ordena por F1 de validación cruzada (solo train, validando con SMS reales). Recién DESPUÉS
# se reporta el test del modelo elegido: así el resultado final no está "elegido a dedo" sobre el test.
# v7: el objetivo de la tesis es un modelo LIGERO que corra en el celular. Por eso se elige el de
# mayor F1 de validación entre los que cumplen un límite de tamaño y de latencia (como referencia,
# Dhanush et al. (2025) llevan a Android un modelo de 145 KB que clasifica un SMS en 1 a 6 ms).
# Con SOLO_LIGEROS = False se elige simplemente el de mayor F1 de validación.

import io

import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline

SOLO_LIGEROS = True
LIMITE_TAMANO_KB, LIMITE_LATENCIA_MS = 1024, 10.0

cols = ["Modelo", "F1 CV (5-fold, media)", "F1 CV (desv. estandar)", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
tabla_final = pd.concat([
    resultados_ganador[cols].assign(Etapa=f"Base ({config_ganadora})"),
    resultados_afinados[cols].assign(Etapa="Grid Search / Ensamble"),
], ignore_index=True).sort_values("F1 CV (5-fold, media)", ascending=False).reset_index(drop=True)
ctx = ctx_ganador  # partición y configuración del modelo ganador


def _modelo_de(fila):
    return (modelos_afinados[fila["Modelo"]] if fila["Etapa"] == "Grid Search / Ensamble"
            else ctx_ganador["modelos_entrenados"][fila["Modelo"]])


# Latencia y tamaño de cada candidato, medidos con SMS de train (el test no se usa para elegir).
medidas = [medir_latencia_y_tamano(_modelo_de(fila), ctx["X_train_txt"], n_muestra=100)
           for _, fila in tabla_final.iterrows()]
tabla_final["Latencia (ms/SMS)"] = [round(latencia, 2) for latencia, _ in medidas]
tabla_final["Tamaño (KB)"] = [round(tamano, 1) for _, tamano in medidas]
tabla_final["¿Cabe en el celular?"] = np.where(
    (tabla_final["Tamaño (KB)"] <= LIMITE_TAMANO_KB) & (tabla_final["Latencia (ms/SMS)"] <= LIMITE_LATENCIA_MS), "Sí", "No")

elegibles = tabla_final[tabla_final["¿Cabe en el celular?"] == "Sí"] if SOLO_LIGEROS else tabla_final
if elegibles.empty:
    print(f"Ningún modelo cumple los límites ({LIMITE_TAMANO_KB} KB y {LIMITE_LATENCIA_MS} ms/SMS); "
          "se elige el de mayor F1 de validación.")
    elegibles = tabla_final
fila_ganadora = elegibles.iloc[0]
mejor_nombre = fila_ganadora["Modelo"]
etapa_ganadora = fila_ganadora["Etapa"]
mejor_modelo = _modelo_de(fila_ganadora)

latencia_final, tamano_final = medir_latencia_y_tamano(mejor_modelo, ctx["X_test_txt"])
y_pred_final = mejor_modelo.predict(ctx["X_test_txt"])
tn, fp, fn, tp = confusion_matrix(ctx["y_test"], y_pred_final).ravel()

if fila_ganadora.name != tabla_final.index[0]:
    mayor_f1 = tabla_final.iloc[0]
    print(f"El de mayor F1 de validación es {mayor_f1['Modelo']} ({mayor_f1['F1 CV (5-fold, media)']:.4f} ± "
          f"{mayor_f1['F1 CV (desv. estandar)']:.4f}), pero pesa {mayor_f1['Tamaño (KB)']:,.0f} KB y tarda "
          f"{mayor_f1['Latencia (ms/SMS)']:.1f} ms/SMS: no cumple los límites para el celular "
          f"({LIMITE_TAMANO_KB} KB y {LIMITE_LATENCIA_MS} ms/SMS).")
regla = "mayor F1 de validación entre los que caben en el celular" if SOLO_LIGEROS else "mayor F1 de validación cruzada"
print(f"Modelo final elegido ({regla}): {mejor_nombre}  ({etapa_ganadora})")
print(f"F1 CV: {fila_ganadora['F1 CV (5-fold, media)']:.4f} ± {fila_ganadora['F1 CV (desv. estandar)']:.4f}")
print(f"\nResultado en el TEST (n={len(ctx['y_test'])}, SMS reales nunca vistos ni usados para elegir):")
print(f"  Accuracy {fila_ganadora['Accuracy']:.4f} | Precision {fila_ganadora['Precision']:.4f} | "
      f"Recall {fila_ganadora['Recall']:.4f} | F1-Score {fila_ganadora['F1-Score']:.4f} | ROC-AUC {fila_ganadora['ROC-AUC']:.4f}")
print(f"  Matriz de confusión: TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"  Latencia extremo a extremo: {latencia_final:.2f} ms/SMS (1 hilo)  |  Tamaño del pipeline: {tamano_final:.1f} KB")

# Con kernel lineal, la SVM decide por el signo de w·x + b: para llevarla al celular basta
# guardar el vector de pesos w en lugar de sus vectores de soporte. LogisticRegression se usa
# solo como contenedor de w y b (predict decide por el mismo signo). El "riesgo %" necesitaría
# además los dos parámetros de calibración de Platt; la clase predicha es idéntica.
# Un ensamble (Voting o Stacking) no es un Pipeline: ahí no aplica.
modelo_svm = mejor_modelo.steps[-1][1] if hasattr(mejor_modelo, "steps") else None
if getattr(modelo_svm, "kernel", None) == "linear":
    solo_pesos = LogisticRegression()
    solo_pesos.coef_ = modelo_svm.coef_.toarray() if hasattr(modelo_svm.coef_, "toarray") else modelo_svm.coef_
    solo_pesos.intercept_, solo_pesos.classes_ = modelo_svm.intercept_, modelo_svm.classes_
    modelo_solo_pesos = Pipeline(mejor_modelo.steps[:-1] + [("modelo", solo_pesos)])
    buffer = io.BytesIO()
    joblib.dump(modelo_solo_pesos, buffer)
    tamano_solo_pesos = buffer.tell() / 1024
    iguales = (modelo_solo_pesos.predict(ctx["X_test_txt"]) == y_pred_final).all()
    print(f"  Guardando solo los pesos (sin los {modelo_svm.support_vectors_.shape[0]} vectores de soporte): "
          f"{tamano_solo_pesos:.1f} KB  |  Mismas predicciones en el test: {iguales}")
print()
tabla_final

#### 5.3.1 ¿Las diferencias entre modelos son significativas?

Varios candidatos quedan a una o dos predicciones de distancia en el test. Para saber si esas diferencias significan algo se calculan intervalos de confianza por bootstrap y el test de McNemar, como en Rasol Al Saidat et al. (2026).

In [ ]:
# --- 5.3.1 ¿Las diferencias son significativas? Intervalos de confianza y test de McNemar ---
# Siguiendo a Rasol Al Saidat et al. (2026): (a) intervalo de confianza del 95 % por bootstrap
# para las métricas del modelo final, y (b) test de McNemar entre el modelo final y cada
# candidato, sobre las MISMAS predicciones del test, con corrección de Holm. Nada de esto se
# usa para elegir el modelo (eso ya se decidió con la validación cruzada): solo dice cuánta
# confianza merecen las diferencias que se ven en el test.
# Las funciones y constantes de esta celda están en src/smishing_peru/estadistica.py.
from smishing_peru.estadistica import holm, intervalo_bootstrap, mcnemar_exacto, _metricas

# (a) Intervalos de confianza del modelo final
y_test = ctx["y_test"]
grupos_test = df["grupo"].values[ctx["idx_test"]]
puntual = _metricas(y_test, y_pred_final)
ic_sms_bajo, ic_sms_alto = intervalo_bootstrap(y_test, y_pred_final)
ic_grp_bajo, ic_grp_alto = intervalo_bootstrap(y_test, y_pred_final, grupos=grupos_test)
tabla_ic = pd.DataFrame({
    "Métrica": list(puntual),
    "Valor en el test": [100 * puntual[m] for m in puntual],
    "IC 95 % (bootstrap por SMS)": [f"{100 * ic_sms_bajo[m]:.2f} - {100 * ic_sms_alto[m]:.2f}" for m in puntual],
    "IC 95 % (bootstrap por grupos)": [f"{100 * ic_grp_bajo[m]:.2f} - {100 * ic_grp_alto[m]:.2f}" for m in puntual],
}).round(2)
print(f"(a) Modelo final ({mejor_nombre}): intervalos de confianza del 95 % (2,000 remuestreos)")
print(tabla_ic.to_string(index=False))

# (b) McNemar: modelo final frente a cada candidato de la tabla de la sección 5.3
todos_los_modelos = {**{n: m for n, m in ctx_ganador["modelos_entrenados"].items()}, **modelos_afinados}
filas = []
for _, fila in tabla_final.iterrows():
    nombre = fila["Modelo"]
    if nombre == mejor_nombre and fila["Etapa"] == etapa_ganadora:
        continue
    pred_otro = todos_los_modelos[nombre].predict(ctx["X_test_txt"])
    b, c, p = mcnemar_exacto(y_test, y_pred_final, pred_otro)
    filas.append({"Candidato": nombre, "Etapa": fila["Etapa"], "F1 test": fila["F1-Score"],
                  "Solo acierta el final": b, "Solo acierta el candidato": c, "p (McNemar exacto)": p})
tabla_mcnemar = pd.DataFrame(filas)
tabla_mcnemar["p (Holm)"] = holm(tabla_mcnemar["p (McNemar exacto)"])
tabla_mcnemar["¿Diferencia significativa (p<0.05)?"] = np.where(tabla_mcnemar["p (Holm)"] < 0.05, "Sí", "No")
print(f"\n(b) Test de McNemar: {mejor_nombre} frente a cada candidato (mismos {len(y_test)} SMS del test)")
print(tabla_mcnemar.round(4).to_string(index=False))

### 5.4 Robustez: aporte de los SMS generados, evaluación por fuente, similitud con train, atajos y particiones

In [ ]:
# --- 5.4 Robustez: ¿cuánto aportan los SMS generados con Gemini? ---
# El test tiene solo SMS adaptados de datasets reales. Se reentrena el modelo final con otros
# conjuntos de entrenamiento y se evalúa en ese mismo test:
#  (a) solo con los SMS reales de train: si el modelo final (reales + generados) lo supera,
#      el data augmentation ayuda;
#  (b) solo con los SMS generados: si funciona, Gemini capturó cómo es el fraude real y no solo
#      su propio estilo (generalización de lo sintético a lo real).
from sklearn.base import clone

from sklearn.base import clone

es_generado_train = df["origen"].values[ctx["idx_train"]] == "generado_llm"
filas = []
for nombre, usar in [("(a) Solo SMS reales de train", ~es_generado_train),
                     ("(b) Solo SMS generados con Gemini", es_generado_train)]:
    modelo_parcial = clone(mejor_modelo).fit(ctx["X_train_txt"][usar], ctx["y_train"][usar])
    filas.append({"Entrenamiento": nombre, "SMS de train": int(usar.sum()),
                  **_metricas(ctx["y_test"], modelo_parcial.predict(ctx["X_test_txt"]))})
filas.append({"Entrenamiento": "Reales + generados (modelo final)", "SMS de train": len(ctx["y_train"]),
              **_metricas(ctx["y_test"], y_pred_final)})
tabla_aporte_generados = pd.DataFrame(filas).round(4)
print(f"Aporte de los SMS generados (test: {len(ctx['y_test'])} SMS adaptados de datasets reales):")
print(tabla_aporte_generados.to_string(index=False))

In [ ]:
# --- (c) Test por FUENTE original del SMS, con sus errores ---
# Responde a "¿en qué datos falla el modelo?": cada fila es una fuente del test.
# FP = HAM marcado como SMISHING (falsa alarma); FN = SMISHING que se escapó.

from sklearn.metrics import accuracy_score

df_test = df.iloc[ctx["idx_test"]].assign(fuente=fuente_detallada(df.iloc[ctx["idx_test"]]).values,
                                          pred=ctx["label_encoder"].inverse_transform(y_pred_final))
filas = []
for fuente_i, g in df_test.groupby("fuente"):
    y_t = (g["label"] == "SMISHING").astype(int)
    y_p = (g["pred"] == "SMISHING").astype(int)
    filas.append({
        "Fuente (test)": NOMBRE_FUENTE.get(fuente_i, fuente_i), "n": len(g),
        "HAM": int((y_t == 0).sum()), "SMISHING": int(y_t.sum()),
        "Accuracy": accuracy_score(y_t, y_p),
        # Con una sola clase en la fuente (p. ej. el refuerzo HAM) el F1 no se define.
        "F1-Score": f1_score(y_t, y_p) if 0 < y_t.sum() < len(y_t) else np.nan,
        "FP": int(((y_t == 0) & (y_p == 1)).sum()), "FN": int(((y_t == 1) & (y_p == 0)).sum()),
    })
tabla_por_fuente = pd.DataFrame(filas).round(4)
print("(c) Modelo final en el test, por fuente original del SMS:")
print(tabla_por_fuente.to_string(index=False))

# Matriz de confusión por origen: muestra si los errores se concentran en un origen.
origenes = sorted(df_test["origen"].unique())
fig, axes = plt.subplots(1, len(origenes), figsize=(5.5 * len(origenes), 4.2))
for ax, origen_i in zip(np.atleast_1d(axes), origenes):
    g = df_test[df_test["origen"] == origen_i]
    cm = confusion_matrix(g["label"], g["pred"], labels=["HAM", "SMISHING"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["HAM", "SMISHING"],
                yticklabels=["HAM", "SMISHING"], ax=ax)
    ax.set_title(f"Test: {origen_i} (n={len(g)})")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

# Los SMS mal clasificados del test, para revisarlos uno por uno.
errores_test = df_test[df_test["label"] != df_test["pred"]][["label", "pred", "fuente", "text"]]
errores_test.to_csv("evidencias/08_errores_del_test.csv", index=False)
print(f"\nSMS mal clasificados en el test: {len(errores_test)} (guardados en evidencias/08_errores_del_test.csv)")
errores_test

In [ ]:
# --- 5.4 (d) ¿El test se parece demasiado a train? Casi-duplicados entre particiones ---
# La partición por grupos evita que dos SMS de una misma llamada a Gemini queden separados,
# pero dos SMS de llamadas distintas pueden seguir pareciéndose mucho (la misma plantilla de
# fraude con otra entidad, o dos SMS originales casi iguales adaptados por separado). Para
# cada SMS del test se mide su similitud coseno con el SMS más parecido de train (n-gramas
# de 3 a 5 caracteres) y se calcula el F1 del modelo final por tramo de similitud. Si el F1
# se mantiene en los SMS poco parecidos a train, el modelo no depende de haber "visto" casi
# el mismo mensaje.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler

vec_caracteres = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
M_train = vec_caracteres.fit_transform([normalizar(t) for t in ctx["X_train_txt"]])
M_test = vec_caracteres.transform([normalizar(t) for t in ctx["X_test_txt"]])
similitud_max = np.asarray((M_test @ M_train.T).max(axis=1).todense()).ravel()

tramos = pd.cut(similitud_max, bins=[0, 0.5, 0.7, 0.9, 1.0001], right=False,
                labels=["< 0.5", "0.5 - 0.7", "0.7 - 0.9", ">= 0.9"])
filas = []
for tramo in tramos.categories:
    m = np.asarray(tramos == tramo)
    if m.sum() == 0:
        continue
    y_t, y_p = ctx["y_test"][m], y_pred_final[m]
    filas.append({"Similitud con el SMS más parecido de train": tramo, "n": int(m.sum()),
                  "% del test": 100 * m.mean(), "SMISHING": int(y_t.sum()),
                  "Accuracy": accuracy_score(y_t, y_p),
                  "F1-Score": f1_score(y_t, y_p) if 0 < y_t.sum() < len(y_t) else np.nan,
                  "Errores": int((y_t != y_p).sum())})
tabla_similitud = pd.DataFrame(filas).round(4)
print("(d) Modelo final en el test, según cuánto se parece cada SMS al más parecido de train:")
print(tabla_similitud.to_string(index=False))
print(f"Similitud mediana: {np.median(similitud_max):.3f}  |  SMS del test con similitud >= 0.9: "
      f"{(similitud_max >= 0.9).sum()} ({(similitud_max >= 0.9).mean():.1%})")


# --- 5.4 (e) Línea base de atajos con la MISMA partición ---
# En 3.2 se midió el F1 de clasificadores triviales con una validación simple. Aquí se
# entrena un árbol de decisión pequeño (profundidad 3) SOLO con señales superficiales en
# train y se evalúa en el mismo test. También se mide el aporte de cada bloque del modelo:
# solo el TF-IDF y solo las 7 variables, con el mismo clasificador del modelo final.
from sklearn.tree import DecisionTreeClassifier

extractor_crudo = StylisticFeatureExtractor(usar_logica_difusa=False)
V_train = extractor_crudo.transform(ctx["X_train_txt"])
V_test = extractor_crudo.transform(ctx["X_test_txt"])
nombres_v = list(extractor_crudo.get_feature_names_out())
filas = []
for nombre, columnas in [("Árbol: solo '¿tiene URL?'", ["tiene_url"]),
                         ("Árbol: solo longitud", ["longitud_msg"]),
                         ("Árbol: solo '¿menciona entidad peruana o soles?'", ["tiene_contexto_peru"]),
                         ("Árbol: las 7 variables (profundidad 3)", nombres_v)]:
    cols = [nombres_v.index(c) for c in columnas]
    arbol = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=RANDOM_STATE)
    y_p = arbol.fit(V_train[:, cols], ctx["y_train"]).predict(V_test[:, cols])
    filas.append({"Modelo": nombre, **_metricas(ctx["y_test"], y_p)})

if hasattr(mejor_modelo, "named_steps"):  # el modelo final es un pipeline simple (no un ensamble)
    clasificador = mejor_modelo.named_steps["modelo"]
    solo_tfidf = Pipeline([
        ("preprocesado", PreprocesadorTexto(quitar_stopwords=ctx["quitar_stopwords"])),
        ("vectorizador", TfidfVectorizer(ngram_range=(1, 2), max_features=3000, min_df=2, token_pattern=r"\S+")),
        ("modelo", clone(clasificador))])
    solo_estilo = Pipeline([
        ("extractor", StylisticFeatureExtractor(usar_logica_difusa=ctx["usar_logica_difusa"])),
        ("escalado", MaxAbsScaler()), ("modelo", clone(clasificador))])
    for nombre, modelo in [("Mismo clasificador, solo TF-IDF", solo_tfidf),
                           ("Mismo clasificador, solo las 7 variables", solo_estilo)]:
        y_p = modelo.fit(ctx["X_train_txt"], ctx["y_train"]).predict(ctx["X_test_txt"])
        filas.append({"Modelo": nombre, **_metricas(ctx["y_test"], y_p)})
filas.append({"Modelo": f"Modelo final completo ({mejor_nombre})", **_metricas(ctx["y_test"], y_pred_final)})
tabla_atajos = pd.DataFrame(filas).round(4)
print("\n(e) Líneas base con la misma partición (test):")
print(tabla_atajos.to_string(index=False))


# --- 5.4 (f) ¿El modelo depende del TIPO de teléfono? ---
# En el corpus, casi todos los SMS con un celular (9XX XXX XXX) son SMISHING y los teléfonos
# fijos aparecen sobre todo en HAM generado: es un sesgo de la generación (los prompts de
# fraude piden "un WhatsApp o celular ficticio" y los avisos legítimos, "el teléfono de la
# central"). En la vida real un amigo también escribe "llámame al 987...". Se mide de 3 formas:
celular_en = lambda textos: np.array([bool(REGEX_CELULAR.search(t)) for t in textos])
tabla_celular = pd.crosstab(df["label"], celular_en(df["text"].values), rownames=["Clase"],
                            colnames=["¿Tiene celular?"])
print("\n(f) SMS con número de celular, por clase (dataset completo):")
print(tabla_celular.to_string())

# (f.1) Prueba contrafactual: a cada HAM del test se le agrega una frase al final. Para saber
# si lo que pesa es el celular o la frase, se compara con la misma frase sin número y con un fijo.
ham_test = ctx["X_test_txt"][ctx["y_test"] == 0]
FRASES = {
    "Sin cambios": "",
    "+ 'Cualquier cosa me avisas.' (sin número)": " Cualquier cosa me avisas.",
    "+ 'Cualquier cosa me llamas al (01) 311 9090.' (fijo)": " Cualquier cosa me llamas al (01) 311 9090.",
    "+ 'Cualquier cosa me llamas al 987 654 321.' (celular)": " Cualquier cosa me llamas al 987 654 321.",
}
ham_variantes = {nombre: [t.rstrip() + frase for t in ham_test] for nombre, frase in FRASES.items()}
ham_con_celular = list(ham_variantes["+ 'Cualquier cosa me llamas al 987 654 321.' (celular)"])
tabla_telefono = pd.DataFrame([{"HAM del test (n = %d)" % len(ham_test): nombre,
                                "% marcados como SMISHING (modelo final)": 100 * mejor_modelo.predict(textos).mean()}
                               for nombre, textos in ham_variantes.items()])
print("\n(f.1) HAM del test a los que se agrega una frase al final:")
print(tabla_telefono.round(2).to_string(index=False))


# (f.2) Mismo modelo, pero sin distinguir celular de fijo (un solo token "telefonotoken").
class PreprocesadorTelefonoUnico(PreprocesadorTexto):
    def transform(self, X):
        return [t.replace("celulartoken", "telefonotoken") for t in super().transform(X)]


if hasattr(mejor_modelo, "named_steps"):
    modelo_tel_unico = clone(mejor_modelo).set_params(
        variables__tfidf__preprocesado=PreprocesadorTelefonoUnico(quitar_stopwords=ctx["quitar_stopwords"]))
    y_p_tel = modelo_tel_unico.fit(ctx["X_train_txt"], ctx["y_train"]).predict(ctx["X_test_txt"])
    tabla_telefono["% marcados (sin distinguir celular de fijo)"] = [
        100 * modelo_tel_unico.predict(textos).mean() for textos in ham_variantes.values()]
    print(f"(f.2) Sin distinguir celular de fijo: F1 test {f1_score(ctx['y_test'], y_p_tel):.4f} "
          f"(modelo final: {f1_score(ctx['y_test'], y_pred_final):.4f})")
    print(tabla_telefono.round(2).to_string(index=False))
    tabla_atajos = pd.concat([tabla_atajos, pd.DataFrame([
        {"Modelo": "Mismo modelo, sin distinguir celular de fijo", **_metricas(ctx["y_test"], y_p_tel)}])],
        ignore_index=True).round(4)


# --- 5.4 (g) ¿El resultado depende de la partición elegida? ---
# Todo el test sale de UNA partición (semilla 42) de los SMS reales. Para saber si el F1 del test es típico o
# si tocó una partición favorable, se repite la partición por grupos con 20 semillas (40 a
# 59, que incluye la 42) y en cada una se reentrena el modelo final con su train y se evalúa
# en su test. La configuración y los hiperparámetros no se vuelven a elegir.
y_todo = (df["label"].values == "SMISHING").astype(int)  # mismo código que LabelEncoder
filas = []
for semilla in range(40, 60):
    i_train, i_test, _, _ = particion_por_grupos(df, seed=semilla)
    modelo_semilla = clone(mejor_modelo).fit(df["text"].values[i_train], y_todo[i_train])
    filas.append({"Semilla de la partición": semilla, "n test": len(i_test),
                  **_metricas(y_todo[i_test], modelo_semilla.predict(df["text"].values[i_test]))})
tabla_particiones = pd.DataFrame(filas).round(4)
f1_particiones = tabla_particiones["F1-Score"]
f1_semilla_42 = f1_particiones[tabla_particiones["Semilla de la partición"] == RANDOM_STATE].iloc[0]
puesto_42 = int((f1_particiones > f1_semilla_42).sum()) + 1
print(f"\n(g) Modelo final reentrenado en 20 particiones distintas (semillas 40 a 59):")
print(f"F1 en el test: media {f1_particiones.mean():.4f} ± {f1_particiones.std():.4f}  |  "
      f"mínimo {f1_particiones.min():.4f}  |  máximo {f1_particiones.max():.4f}")
lado = "por encima" if f1_semilla_42 > f1_particiones.mean() else "por debajo"
print(f"La partición usada (semilla {RANDOM_STATE}, F1 {f1_semilla_42:.4f}) queda en el puesto {puesto_42} de 20, "
      f"{lado} de la media: el valor típico que cabe esperar es la media.")

### 5.5 Robustez adversaria

Se alteran los SMISHING del test con ataques de caracteres y de palabras, como en la prueba adversaria de Salinas Bolaños y Ticona (2025), y se mide cuántos se siguen detectando, con y sin la defensa por desofuscación.

In [ ]:
# --- 5.5 Robustez adversaria: ¿el modelo resiste SMS alterados para evadirlo? ---
# Salinas Bolaños y Ticona (2025) evaluaron su modelo con fraudes alterados por ataques
# adversarios, y Bajaj y Vishwakarma (2023) y Salman et al. (2024) muestran que los cambios
# de caracteres (sobre todo el espaciado entre letras) evaden a muchos clasificadores. Aquí
# se alteran los SMISHING del test con 6 ataques aplicados a las 3 palabras del SMS que más
# empujan al modelo hacia SMISHING (el peor caso: el atacante sabe qué palabras delatan el
# fraude) y se mide cuántos se siguen detectando (recall). Después se repite con la
# variante robusta del mismo modelo (desofuscar() antes del preprocesado, sección 4),
# entrenada con el mismo train. El HAM no se altera: al estafador solo le interesa que su
# SMISHING pase como HAM.
# Las funciones y constantes de esta celda están en src/smishing_peru/ataques.py.
from smishing_peru.ataques import ATAQUES, atacar, palabras_clave_del_fraude

es_smishing = ctx["y_test"] == 1
smishing_test = ctx["X_test_txt"][es_smishing]
spans_por_sms = [palabras_clave_del_fraude(mejor_modelo, t) for t in smishing_test]

rng = random.Random(RANDOM_STATE)
sms_atacados = {"Sin ataque": list(smishing_test)}
for nombre, funcion in ATAQUES.items():
    sms_atacados[nombre] = [atacar(t, s, funcion, rng) for t, s in zip(smishing_test, spans_por_sms)]

# Variante robusta: el mismo modelo, con desofuscar() en el preprocesado y en el extractor.
modelo_robusto = fijar_parametro(mejor_modelo, "robusto", True).fit(ctx["X_train_txt"], ctx["y_train"])
y_pred_robusto = modelo_robusto.predict(ctx["X_test_txt"])

filas = []
for nombre, textos in sms_atacados.items():
    filas.append({"Ataque (sobre los SMISHING del test)": nombre, "n": len(textos),
                  "Recall modelo final": mejor_modelo.predict(textos).mean(),
                  "Recall variante robusta": modelo_robusto.predict(textos).mean()})
tabla_adversaria = pd.DataFrame(filas)
tabla_adversaria["Caída modelo final"] = tabla_adversaria["Recall modelo final"] - tabla_adversaria["Recall modelo final"].iloc[0]
tabla_adversaria["Caída variante robusta"] = (tabla_adversaria["Recall variante robusta"]
                                              - tabla_adversaria["Recall variante robusta"].iloc[0])
print("Recall (proporción de SMISHING detectados) antes y después de cada ataque:")
print(tabla_adversaria.round(4).to_string(index=False))

limpio_final, limpio_robusto = _metricas(ctx["y_test"], y_pred_final), _metricas(ctx["y_test"], y_pred_robusto)
print("\nEn el test SIN alterar (HAM + SMISHING), ¿la defensa cuesta algo?")
print(pd.DataFrame([limpio_final, limpio_robusto], index=["Modelo final", "Variante robusta"]).round(4).to_string())

print("\nEjemplo de cada ataque sobre el mismo SMS:")
ejemplo = next(i for i, s in enumerate(spans_por_sms) if len(s) == 3)
for nombre, textos in sms_atacados.items():
    print(f"  {nombre}: {textos[ejemplo]}")

### 5.6 Validación externa con SMS peruanos reales (opcional)

In [ ]:
# --- 5.6 Validación externa con SMS peruanos reales (opcional, muy recomendada) ---
# Todo el dataset pasó por un LLM (adaptado o generado). La prueba más fuerte es
# evaluar el modelo YA ENTRENADO con SMS reales recibidos en el Perú, que nunca
# pasaron por Gemini. Formato: 'sms_reales_peru.csv' con columnas text,label
# (label = HAM o SMISHING). Fuentes: SMS propios y de familiares, capturas de alertas
# públicas de bancos, Indecopi o la PNP. Anonimiza nombres y números antes de usarlos.
import os

from sklearn.metrics import precision_score, recall_score

ARCHIVO_EXTERNO = "sms_reales_peru.csv"
if os.path.exists(ARCHIVO_EXTERNO):
    externos = pd.read_csv(ARCHIVO_EXTERNO).dropna(subset=["text", "label"])
    externos["label"] = externos["label"].str.upper().str.strip()
    y_ext = ctx["label_encoder"].transform(externos["label"])
    y_ext_pred = mejor_modelo.predict(externos["text"].values)  # el pipeline recibe el SMS en texto plano
    print(f"Validación externa con {len(externos)} SMS reales:")
    print(f"  Accuracy {accuracy_score(y_ext, y_ext_pred):.4f} | Precision {precision_score(y_ext, y_ext_pred, zero_division=0):.4f} | "
          f"Recall {recall_score(y_ext, y_ext_pred, zero_division=0):.4f} | F1 {f1_score(y_ext, y_ext_pred, zero_division=0):.4f}")
    print("  Matriz de confusión [[TN, FP], [FN, TP]]:", confusion_matrix(y_ext, y_ext_pred).tolist())
else:
    print(f"No se encontró '{ARCHIVO_EXTERNO}': se omite la validación externa.")
    print("Sube un CSV con columnas text,label (HAM/SMISHING) con SMS reales para activarla.")

### 5.6 (b) Prueba extra con smishing real en español (IMC25)

IMC25 (Agarwal et al., 2025) reúne SMS de smishing que los usuarios reportaron en foros públicos, leídos de capturas de pantalla. Tiene 4,639 en español, la mayoría de España, según las marcas que suplantan, y **no tiene SMS legítimos**. Por eso aquí solo se mide qué porcentaje de esos fraudes reales detecta el modelo (sensibilidad). IMC25 no se usa para entrenar ni para elegir nada. Licencia CC BY 4.0.

In [ ]:
# --- 5.6 (b) Prueba extra con smishing real en español (IMC25) ---
# IMC25 (Agarwal et al., 2025; CC BY 4.0) no tiene SMS legítimos: solo mide la sensibilidad, es
# decir, qué porcentaje de esos fraudes reales detecta el modelo final. No se usa para entrenar.
# Los autores taparon los datos sensibles con etiquetas (<URL>, <NAMED_ENTITY>, <PHONE_NUMBER>...):
#  - <URL> se cambia por "http://enlace", para que el modelo sepa que ahí había un enlace (el
#    preprocesamiento lo convierte en urltoken, como a cualquier enlace);
#  - las demás etiquetas se quitan.
# El reconocedor de entidades de los autores también tapó palabras comunes ("ha <NAMED_ENTITY>
# temporalmente"), así que se reporta aparte el subconjunto sin <NAMED_ENTITY>, el menos alterado.
# Se excluyen los SMS de tipo "spam" (publicidad, que no siempre es fraude) y los textos repetidos.
import os
import urllib.request

URL_IMC25 = "https://raw.githubusercontent.com/reportsmishing/Smishing-Dataset-IMC25/main/dataset/final_dataset_output.csv"
ARCHIVO_IMC25 = "imc25_final_dataset_output.csv"
LATAM = {"ARG", "BOL", "CHL", "COL", "CRI", "CUB", "DOM", "ECU", "GTM", "HND", "MEX", "NIC", "PAN", "PRY", "SLV",
         "URY", "VEN"}

if not os.path.exists(ARCHIVO_IMC25):
    try:
        urllib.request.urlretrieve(URL_IMC25, ARCHIVO_IMC25)
    except Exception as e:
        print(f"No se pudo descargar IMC25 ({e}). Puedes subirlo a mano con el nombre '{ARCHIVO_IMC25}'.")

tabla_imc25 = None
if os.path.exists(ARCHIVO_IMC25):
    imc = pd.read_csv(ARCHIVO_IMC25)
    imc = imc[(imc["language"] == "Spanish") & (imc["scam_type"] != "spam")].copy()
    n_sin_spam = len(imc)
    imc["entidad_tapada"] = imc["text"].str.contains("<NAMED_ENTITY>", regex=False)
    imc["text_limpio"] = (imc["text"].str.replace("<URL>", " http://enlace ", regex=False)
                          .str.replace(r"<[A-Z_]+>", " ", regex=True)
                          .str.replace(r"\s+", " ", regex=True).str.strip())
    imc = imc[imc["text_limpio"].str.len() >= 20].drop_duplicates("text_limpio").reset_index(drop=True)
    pais = imc["original_network_country"]
    imc["region"] = np.select([pais == "ESP", pais == "PER", pais.isin(LATAM), pais.isna()],
                              ["España", "Perú", "Otros de Latinoamérica", "Sin dato"], "Otros países")
    clase_fraude = ctx["label_encoder"].transform(["SMISHING"])[0]
    imc["detectado"] = mejor_modelo.predict(imc["text_limpio"].values) == clase_fraude

    def _fila(nombre, parte):
        return {"Subconjunto": nombre, "n": len(parte), "Detectados": int(parte["detectado"].sum()),
                "Sensibilidad (%)": round(100 * parte["detectado"].mean(), 2) if len(parte) else np.nan}

    filas = [_fila("Todos", imc),
             _fila("Sin <NAMED_ENTITY> (menos alterados)", imc[~imc["entidad_tapada"]]),
             _fila("Con <NAMED_ENTITY>", imc[imc["entidad_tapada"]])]
    filas += [_fila(f"Región: {r}", imc[imc["region"] == r])
              for r in ["España", "Perú", "Otros de Latinoamérica", "Otros países", "Sin dato"]]
    filas += [_fila(f"Tipo: {t}", imc[imc["scam_type"] == t]) for t in imc["scam_type"].value_counts().index]
    tabla_imc25 = pd.DataFrame(filas)

    print(f"IMC25 en español: {n_sin_spam} SMS de fraude sin el spam; {len(imc)} tras quitar los repetidos "
          f"y los de menos de 20 caracteres.")
    print(f"Sensibilidad del modelo final ({mejor_nombre}) en IMC25: {imc['detectado'].mean():.2%} "
          f"(en el test propio: {recall_score(ctx['y_test'], y_pred_final):.2%}).\n")
    print(tabla_imc25.to_string(index=False))
    print("\nEjemplos de fraudes que el modelo no detectó:")
    for texto in imc.loc[~imc["detectado"], "text_limpio"].sample(min(5, (~imc["detectado"]).sum()),
                                                                  random_state=RANDOM_STATE):
        print(f"  - {texto[:160]}")

### 5.6 (c) Prueba extra: ¿cuánto mejora con fraudes reales?

En 5.6 (b) el modelo detecta una parte pequeña de los fraudes reales de IMC25. Para medir cuánto ayudaría tener fraudes reales en el entrenamiento, IMC25 se divide en dos mitades y el modelo final se reentrena con su mismo train más la mitad A. Luego se comparan los dos modelos en la mitad B, que ninguno vio, y en el test propio, para ver si aparecen falsas alarmas.

Los SMS casi iguales (la misma campaña con otro nombre o enlace) quedan siempre en la misma mitad. **Esta prueba no cambia el modelo final ni ningún resultado anterior.**

In [ ]:
# --- 5.6 (c) ¿Cuánto mejora con fraudes reales? Reentrenando con la mitad de IMC25 ---
# Prueba extra: NO cambia el modelo final ni ningún resultado anterior.
# IMC25 se divide en dos mitades por grupos: los SMS casi iguales (coseno >= 0.8 en n-gramas de
# 3 a 5 caracteres, como en 5.4 d), que suelen ser la misma campaña con otro nombre o enlace,
# quedan siempre en la misma mitad. El modelo final se reentrena con su mismo train más la
# mitad A de IMC25 (como SMISHING) y se compara con el modelo final en:
#  - la mitad B de IMC25, fraudes reales que ninguno de los dos vio;
#  - el test propio (347 SMS), para ver si al agregar esos fraudes aparecen falsas alarmas.
from scipy.sparse.csgraph import connected_components
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit

tabla_imc25_entrenando = None
if tabla_imc25 is not None:
    vec_imc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
    M_imc = vec_imc.fit_transform([normalizar(t) for t in imc["text_limpio"]])
    _, grupo_imc = connected_components((M_imc @ M_imc.T) >= 0.8, directed=False)
    idx_a, idx_b = next(GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=RANDOM_STATE)
                        .split(imc, groups=grupo_imc))
    imc_a, imc_b = imc.iloc[idx_a], imc.iloc[idx_b]

    modelo_con_imc = clone(mejor_modelo).fit(
        np.concatenate([np.asarray(ctx["X_train_txt"], dtype=object), imc_a["text_limpio"].values]),
        np.concatenate([np.asarray(ctx["y_train"]), np.full(len(imc_a), clase_fraude)]))

    # Similitud de cada SMS de la mitad B con el más parecido de la mitad A
    sim_b = np.asarray((M_imc[idx_b] @ M_imc[idx_a].T).max(axis=1).todense()).ravel()
    partes_b = {
        "IMC25 mitad B: todos": np.ones(len(imc_b), bool),
        "IMC25 mitad B: sin <NAMED_ENTITY>": ~imc_b["entidad_tapada"].values,
        "IMC25 mitad B: 60 caracteres o menos": (imc_b["text_limpio"].str.len() <= 60).values,
        "IMC25 mitad B: poco parecidos a la mitad A (< 0.5)": sim_b < 0.5,
    }
    y_test_con_imc = modelo_con_imc.predict(ctx["X_test_txt"])
    filas = []
    for nombre, m in partes_b.items():
        fila = {"Medida": f"{nombre} (n = {m.sum()}), sensibilidad"}
        for col, modelo in [("Modelo final (sin IMC25)", mejor_modelo), ("Modelo final + mitad A de IMC25", modelo_con_imc)]:
            fila[col] = round(100 * (modelo.predict(imc_b["text_limpio"].values[m]) == clase_fraude).mean(), 2)
        filas.append(fila)
    for nombre, fn in [("exactitud", accuracy_score), ("precisión", precision_score),
                       ("sensibilidad", recall_score), ("F1", f1_score)]:
        filas.append({"Medida": f"Test propio (n = {len(ctx['y_test'])}): {nombre}",
                      "Modelo final (sin IMC25)": round(100 * fn(ctx["y_test"], y_pred_final), 2),
                      "Modelo final + mitad A de IMC25": round(100 * fn(ctx["y_test"], y_test_con_imc), 2)})
    n_ham_test = int((ctx["y_test"] != clase_fraude).sum())
    filas.append({"Medida": f"Test propio: SMS legítimos marcados como fraude (de {n_ham_test})",
                  "Modelo final (sin IMC25)": int(((y_pred_final == clase_fraude) & (ctx["y_test"] != clase_fraude)).sum()),
                  "Modelo final + mitad A de IMC25": int(((y_test_con_imc == clase_fraude) & (ctx["y_test"] != clase_fraude)).sum())})
    tabla_imc25_entrenando = pd.DataFrame(filas, dtype=object)  # object: los conteos se ven como enteros

    print(f"IMC25 dividido por grupos de SMS casi iguales: mitad A {len(imc_a)} SMS (para entrenar), "
          f"mitad B {len(imc_b)} SMS (para evaluar).")
    print(f"SMS de la mitad B con un SMS casi igual (>= 0.8) en la mitad A: {(sim_b >= 0.8).sum()} (deben ser 0)\n")
    print(tabla_imc25_entrenando.to_string(index=False))
else:
    print("IMC25 no está disponible (ver 5.6 b): se omite esta prueba.")

### 5.7 Curvas de entrenamiento y validación

Estas curvas sirven para revisar si el modelo está sobreajustado (overfitting): (a) el F1 a medida que crece la cantidad de datos de entrenamiento y (b) la pérdida de XGBoost después de cada árbol.

In [ ]:
# --- 5.7 (a) Curva de aprendizaje del modelo final: ¿hay sobreajuste (overfitting)? ---
# Se entrena el modelo final con porciones crecientes de train (10% ... 100%) y en cada
# una se mide el F1 en los SMS con los que entrenó (curva de entrenamiento) y en SMS que
# no vio (curva de validación: pliegues de SMS reales de train; el test no se usa).
#  - Si las dos curvas terminan juntas y altas: el modelo generaliza.
#  - Si la de entrenamiento queda muy arriba de la de validación: sobreajuste.
#  - Si la de validación sigue subiendo al final: más datos todavía ayudarían.
from sklearn.model_selection import learning_curve

import time

FOLDS_CURVA = 3  # el Stacking es costoso: 3 pliegues x 5 tamaños = 15 entrenamientos (~10-20 min en Colab)
t0 = time.perf_counter()
n_entrenamiento, f1_train, f1_val = learning_curve(
    clone(mejor_modelo), ctx["X_train_txt"], ctx["y_train"], groups=ctx["grupos_train"],
    cv=ValidacionSoloReales(n_splits=FOLDS_CURVA),
    train_sizes=np.linspace(0.1, 1.0, 5), scoring="f1", n_jobs=-1, shuffle=True, random_state=RANDOM_STATE)
print(f"Curva de aprendizaje calculada en {time.perf_counter() - t0:.0f} s")

tabla_curva = pd.DataFrame({
    "SMS de entrenamiento": n_entrenamiento,
    "F1 entrenamiento": f1_train.mean(axis=1), "F1 validación": f1_val.mean(axis=1),
    "F1 validación (desv.)": f1_val.std(axis=1),
})
tabla_curva["Brecha (entrenamiento - validación)"] = tabla_curva["F1 entrenamiento"] - tabla_curva["F1 validación"]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for valores, nombre, color in [(f1_train, "Entrenamiento", "#4C72B0"), (f1_val, "Validación", "#DD8452")]:
    media, desv = valores.mean(axis=1), valores.std(axis=1)
    ax.plot(n_entrenamiento, media, "o-", color=color, label=nombre)
    ax.fill_between(n_entrenamiento, media - desv, media + desv, color=color, alpha=0.15)
ax.set_xlabel("SMS usados para entrenar")
ax.set_ylabel("F1-Score")
ax.set_title(f"Curva de aprendizaje - {mejor_nombre}")
ax.set_ylim(max(0.0, f1_val.mean(axis=1).min() - 0.05), 1.005)
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

brecha = tabla_curva["Brecha (entrenamiento - validación)"].iloc[-1]
mejora_final = tabla_curva["F1 validación"].iloc[-1] - tabla_curva["F1 validación"].iloc[-2]
if brecha < 0.02:
    lectura = "brecha pequeña: el modelo generaliza a SMS que no vio; no hay sobreajuste relevante."
elif brecha < 0.05:
    lectura = "brecha moderada: hay un sobreajuste leve."
else:
    lectura = "brecha grande: el modelo está sobreajustado (memoriza el entrenamiento)."
print(f"Con todo train: F1 entrenamiento {tabla_curva['F1 entrenamiento'].iloc[-1]:.4f} | "
      f"F1 validación {tabla_curva['F1 validación'].iloc[-1]:.4f} | brecha {brecha:.4f} -> {lectura}")
print(f"Último tramo de la curva de validación: {mejora_final:+.4f} "
      + ("(sigue subiendo: más datos ayudarían)" if mejora_final > 0.005 else "(ya se estabilizó)"))
tabla_curva.round(4)

In [ ]:
# --- 5.7 (b) Curva de entrenamiento por iteración (XGBoost afinado) ---
# XGBoost entrena árbol por árbol, así que se puede ver la pérdida (log-loss) y el error
# en entrenamiento y en validación después de cada árbol, como las curvas por época de
# una red neuronal. Validación = un pliegue de SMS reales de train (el test no se usa).
from xgboost import XGBClassifier

from xgboost import XGBClassifier

xgb_pipeline = modelos_afinados["XGBoost afinado (Grid Search)"]
xgb_base = xgb_pipeline.named_steps["modelo"]
arboles_elegidos = xgb_base.get_params()["n_estimators"]
cv_split = ValidacionSoloReales(n_splits=5)
i_tr, i_va = next(cv_split.split(ctx["X_train_txt"], ctx["y_train"], groups=ctx["grupos_train"]))
# Las variables (TF-IDF + 7 variables) se ajustan solo con la parte de entrenamiento del pliegue.
variables_curva = clone(xgb_pipeline.named_steps["variables"]).fit(ctx["X_train_txt"][i_tr])
X_tr_curva = variables_curva.transform(ctx["X_train_txt"][i_tr])
X_va_curva = variables_curva.transform(ctx["X_train_txt"][i_va])
xgb_curva = XGBClassifier(**{**xgb_base.get_params(), "n_estimators": max(400, 2 * arboles_elegidos),
                             "eval_metric": ["logloss", "error"], "n_jobs": -1})
xgb_curva.fit(X_tr_curva, ctx["y_train"][i_tr],
              eval_set=[(X_tr_curva, ctx["y_train"][i_tr]), (X_va_curva, ctx["y_train"][i_va])],
              verbose=False)
historial = xgb_curva.evals_result()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metrica, titulo in [(axes[0], "logloss", "Pérdida (log-loss)"), (axes[1], "error", "Error (1 - accuracy)")]:
    ax.plot(historial["validation_0"][metrica], label="Entrenamiento", color="#4C72B0")
    ax.plot(historial["validation_1"][metrica], label="Validación", color="#DD8452")
    ax.axvline(arboles_elegidos, color="gray", linestyle="--", lw=1, label=f"Árboles elegidos ({arboles_elegidos})")
    ax.set_xlabel("Número de árboles (iteraciones)")
    ax.set_title(f"XGBoost: {titulo}")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

perdida_val = np.array(historial["validation_1"]["logloss"])
print(f"Log-loss de validación con {arboles_elegidos} árboles: {perdida_val[arboles_elegidos - 1]:.4f} | "
      f"mínimo {perdida_val.min():.4f} en el árbol {perdida_val.argmin() + 1}")
print("Si la pérdida de validación empezara a subir mientras la de entrenamiento sigue bajando, "
      "sería sobreajuste; el número de árboles elegido debe quedar antes de ese punto.")

### 5.8 Comparación con el estado del arte

(a) Resultados que reportan los artículos de la revisión (no son directamente comparables: cada estudio usa otro dataset, otro idioma y otra partición; la prueba de Salinas Bolaños y Ticona es adversaria). (b) Réplica del mismo pipeline en los datasets originales en inglés, con una partición 80/20 estratificada.

In [ ]:
# --- 5.8 (a) Comparación con el estado del arte ---
# Cifras de la matriz de revisión del estado del arte (ESTADO_ARTE_TakedaHiroshi), en %.
# NaN = el artículo no reporta esa métrica. OJO: no son directamente comparables, porque
# cada estudio usa otro dataset, otro idioma y otra partición. La comparación justa,
# con los mismos datasets y la misma división, está en (b).
ESTADO_DEL_ARTE = [
    # (Estudio, Idioma / contexto, Dataset, Técnica, Accuracy, Precision, Recall, F1)
    # OJO: su prueba es ADVERSARIA (los 454 fraudes del test fueron alterados con 6 ataques).
    ("Salinas Bolaños & Ticona (2025)", "Español (Perú)",
     "4,475 SMS: 1,740 reales del Perú (capturas + OCR) + Dataset_5971 y Smishtank traducidos; prueba adversaria",
     "RNN 3 capas + FastText", 85.62, 84.49, 88.77, 86.57),
    ("Birthriya et al. (2026)", "Inglés", "UCI + 545 smishing de Smishtank (6,119)",
     "AdaBoost + TF-IDF + variables difusas", 99.10, 99.30, 94.00, 96.58),
    ("Abdul Samad et al. (2024), SMS-Shield", "Inglés", "SMS Phishing (Mendeley) + UCI, con sobremuestreo",
     "Random Forest", 99.05, 98.49, 99.42, 98.95),
    ("Abdul Samad et al. (2023), SmishGuard", "Inglés", "SMS Phishing (Mendeley) + UCI (7,261)",
     "XGBoost + TF-IDF + LDA", 98.47, 97.91, 95.20, 96.52),
    # Xu et al. reportan precisión, sensibilidad y F1 sin decimales (99 %).
    ("Xu, Qadir & Sadiq (2025)", "Inglés", "SMS Phishing (Mendeley) + UCI (11,545), SMOTE",
     "Ensamble SVM + RF", 99.58, 99.0, 99.0, 99.0),
    ("Aşkın & Baykara (2025)", "Inglés", "SMS Phishing (Mendeley): 638 smishing / 4,844 ham",
     "BERT afinado", 99.82, 100.0, 98.41, 99.20),
    ("Ahammad et al. (2025)", "Inglés", "UCI SMS Spam Collection", "SVM + TF-IDF", 97.94, 97.04, 87.33, 91.93),
    ("Rai (2025)", "Filipinas", "Philippine Spam/Scam + Mishra & Soni + propios (6,303)",
     "SVM lineal + 12 rasgos manuales", 97.46, 96.44, 92.49, 94.43),
    ("Ibrahim et al. (2024)", "Árabe (traducido)", "Mishra & Soni traducido al árabe",
     "Random Forest + TF-IDF", 98.66, 99.10, 98.23, 98.67),
    ("Mambina et al. (2022)", "Suajili", "32,259 SMS reales de Tanzania", "Random Forest + TF-IDF",
     99.86, np.nan, np.nan, 99.86),
    ("Kaur & Jain (2025)", "Inglés", "Mendeley + SMS Smishing Collection (6,695)",
     "LSTM + Random Forest (rasgos de URL)", 91.19, np.nan, np.nan, np.nan),
    ("Cho & Seo (2025)", "Coreano", "300,472 muestras (SMS y transcripciones de llamadas)",
     "KoBERT + LoRA + TAPT", 84.51, np.nan, np.nan, np.nan),
]
COLUMNAS_EA = ["Estudio", "Idioma / contexto", "Dataset", "Técnica", "Accuracy", "Precision", "Recall", "F1-Score"]
este_trabajo = ("ESTE TRABAJO", "Español (Perú)", f"Peruano ({len(df):,} SMS), test de {len(ctx['y_test'])} SMS reales",
                mejor_nombre, *(round(100 * fila_ganadora[m], 2) for m in ["Accuracy", "Precision", "Recall", "F1-Score"]))
tabla_estado_arte = pd.DataFrame([este_trabajo] + ESTADO_DEL_ARTE, columns=COLUMNAS_EA)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, metrica in zip(axes, ["Accuracy", "F1-Score"]):
    datos = tabla_estado_arte.dropna(subset=[metrica]).sort_values(metrica)
    colores = ["#C44E52" if e == "ESTE TRABAJO" else "#8C9DB5" for e in datos["Estudio"]]
    ax.barh(datos["Estudio"], datos[metrica], color=colores)
    for y_i, v in enumerate(datos[metrica]):
        ax.text(v + 0.2, y_i, f"{v:.2f}", va="center", fontsize=8)
    ax.set_xlim(max(0, datos[metrica].min() - 5), 101.5)
    ax.set_title(f"{metrica} (%) reportado")
plt.tight_layout()
plt.show()
tabla_estado_arte

In [ ]:
# --- 5.8 (b) Réplica en los datasets originales: comparación justa con los autores ---
# Se corre EXACTAMENTE el mismo pipeline (TF-IDF + 7 variables + modelo final, con sus
# hiperparámetros) sobre los datasets en inglés que usaron los autores, con la división
# más usada en esos estudios (80/20 estratificada, semilla 42). Si aquí el pipeline
# rinde como ellos, el diseño está bien; y la diferencia con el F1 en español peruano
# muestra cuánto cuesta el cambio de idioma y de contexto.
# Las stopwords no se quitan (la lista es del español) y la variable "contexto peruano"
# casi siempre vale 0 en inglés: se deja igual para no cambiar el pipeline. OJO: cada estudio
# usó su propia partición; aquí se usa 80/20 estratificada para todos, así que la
# comparación es aproximada.
from sklearn.model_selection import train_test_split

modelo_replica = fijar_parametro(mejor_modelo, "quitar_stopwords", False)  # mismo pipeline, sin quitar stopwords

uci_original = cargar_uci_sms("SMSSpamCollection.csv")
d5971_original = cargar_dataset_5971("Dataset_5971.csv")
DATASETS_ORIGINALES = {
    # nombre: (dataset, estudio de referencia con ese mismo dataset, sus métricas %)
    "UCI SMS Spam Collection (ham vs spam)": (
        uci_original, "Ahammad et al. (2025), SVM", {"Accuracy": 97.94, "Precision": 97.04, "Recall": 87.33}),
    "Dataset_5971 (ham vs smishing, sin el spam)": (
        d5971_original[d5971_original["label_original"] != "spam"], "Aşkın & Baykara (2025), BERT",
        {"Accuracy": 99.82, "Precision": 100.0, "Recall": 98.41, "F1-Score": 99.20}),
    "Dataset_5971 + UCI (ham vs spam y smishing)": (
        pd.concat([d5971_original, uci_original]), "Xu, Qadir & Sadiq (2025), SVM + RF",
        {"Accuracy": 99.58, "Precision": 99.0, "Recall": 99.0, "F1-Score": 99.0}),
}

filas = []
for nombre, (datos, estudio, ref) in DATASETS_ORIGINALES.items():
    datos = datos.dropna(subset=["text", "label"]).drop_duplicates(subset="text").reset_index(drop=True)
    y_orig = (datos["label"] == "SMISHING").astype(int).values
    i_tr, i_te = train_test_split(np.arange(len(datos)), test_size=0.2, stratify=y_orig, random_state=RANDOM_STATE)
    t0 = time.perf_counter()
    y_p = clone(modelo_replica).fit(datos["text"].values[i_tr], y_orig[i_tr]).predict(datos["text"].values[i_te])
    print(f"{nombre}: {len(datos)} SMS ({y_orig.sum()} positivos), entrenado en {time.perf_counter() - t0:.0f} s")
    filas.append({
        "Dataset": nombre, "n": len(datos),
        "Accuracy": 100 * accuracy_score(y_orig[i_te], y_p), "Precision": 100 * precision_score(y_orig[i_te], y_p),
        "Recall": 100 * recall_score(y_orig[i_te], y_p), "F1-Score": 100 * f1_score(y_orig[i_te], y_p),
        "Estudio de referencia": estudio,
        **{f"{m} ref.": ref.get(m, np.nan) for m in ["Accuracy", "Precision", "Recall", "F1-Score"]},
    })
tabla_replica = pd.DataFrame(filas).round(2)
tabla_replica

## 6. Inferencia en Vivo y Diagnóstico Interactivo



In [ ]:
# Inferencia en vivo y diagnóstico interactivo
# Las funciones y constantes de esta celda están en src/smishing_peru/diagnostico.py.
from functools import partial

from smishing_peru.diagnostico import evaluar_sms_peruano, _gatillo_por_palabras

# Gatillo psicológico con un clasificador pequeño (TF-IDF + regresión logística) entrenado
# con las etiquetas de gatillo de los SMISHING de train: el gatillo pedido en cada llamada
# de generación o el que asignó el LLM al adaptar. Solo explica el resultado al usuario; no
# interviene en la decisión SMISHING/HAM. Se evalúa con los SMISHING del test.
es_smishing_df = (df["label"] == "SMISHING").values
idx_gatillo_train = [i for i in ctx["idx_train"] if es_smishing_df[i]]
idx_gatillo_test = [i for i in ctx["idx_test"] if es_smishing_df[i]]
modelo_gatillo = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=partial(preprocesar_texto, quitar_stopwords=False),
                              token_pattern=r"\S+", ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
]).fit(df["text"].values[idx_gatillo_train], df["trigger"].values[idx_gatillo_train])

gatillo_real = df["trigger"].values[idx_gatillo_test]
gatillo_palabras = np.array([_gatillo_por_palabras(normalizar(t)) or "ninguno"
                             for t in df["text"].values[idx_gatillo_test]])
gatillo_modelo = modelo_gatillo.predict(df["text"].values[idx_gatillo_test])
clases_gatillo = sorted(set(gatillo_real))
tabla_gatillo = pd.DataFrame([
    {"Detección del gatillo (SMISHING del test, n = %d)" % len(gatillo_real): nombre,
     "Exactitud": accuracy_score(gatillo_real, pred),
     "F1 macro": f1_score(gatillo_real, pred, average="macro", labels=clases_gatillo),
     "% sin gatillo": 100 * (pred == "ninguno").mean()}
    for nombre, pred in [("Palabras clave (v4)", gatillo_palabras), ("Regresión logística (v5)", gatillo_modelo)]
]).round(4)
print(tabla_gatillo.to_string(index=False))

In [ ]:
# Pruebas con mensajes peruanos nuevos (escritos a mano, no vistos durante el entrenamiento).
# Incluye "casos difíciles": notificaciones LEGÍTIMAS de bancos con teléfono de la central y
# fraudes SIN enlace, como el del familiar que "cambió de número".
pruebas = [
    # (SMS, clase esperada)
    ("BCP: Su cuenta ha sido BLOQUEADA por seguridad. Verifique urgente en bcp-verifica.info/x8k2", "SMISHING"),
    ("SUNAT: Usted registra una deuda pendiente de S/ 480. Evite el embargo en sunat-tramite.xyz/91a2", "SMISHING"),
    ("Hola ma, se me malogro el cel y este es mi nuevo numero. Yapeame S/ 200 porfa, es urgente, luego te explico", "SMISHING"),
    ("SAT Lima: tiene una papeleta con 70% de descuento SOLO HOY. Pague aqui sat-descuentos.online/p7", "SMISHING"),
    ("Olva Courier: su paquete no pudo entregarse por direccion incompleta. Llame al 01-7094512 para reprogramar", "SMISHING"),
    ("Trabaja desde casa dando likes y gana S/ 150 diarios. Escribenos al WhatsApp +51 987 654 321", "SMISHING"),
    ("Tio, soy Diego, me robaron el celular y este es mi numero nuevo. Plineame S/ 150 para el taxi, manana te devuelvo", "SMISHING"),
    ("Yape: Recibiste S/ 50.00 de Carlos Ramos.", "HAM"),
    ("Interbank: Realizaste un consumo de S/ 45.90 en Tottus con tu tarjeta ****1234. Si no lo reconoces llama al 311-9090.", "HAM"),
    ("BCP: Transferiste S/ 300.00 a Maria Lopez desde tu cuenta ****5678. Si no reconoces esta operacion llama al (01) 311-9898.", "HAM"),
    ("Tu codigo de verificacion BBVA es 482913. No lo compartas con nadie, ni con personal del banco.", "HAM"),
    ("Hola profe, disculpe la tardanza, ya estoy llegando a la reunion", "HAM"),
    ("Oe causa, manana pichanga a las 8 en la losa de Surco, confirma pe", "HAM"),
]

filas = []
for sms, esperado in pruebas:
    r = evaluar_sms_peruano(sms, mejor_modelo, ctx["label_encoder"], modelo_gatillo)
    predicho = "SMISHING" if "SMISHING" in r["clasificacion"] else "HAM"
    filas.append({"SMS": sms, "Esperado": esperado, "Predicho": predicho,
                  "Riesgo %": r["riesgo_pct"], "Acierto": predicho == esperado})

tabla_pruebas = pd.DataFrame(filas)
print(f"Aciertos en las pruebas a mano: {tabla_pruebas['Acierto'].sum()} de {len(tabla_pruebas)}")
tabla_pruebas

## 7. Visualización de Resultados



In [ ]:
# Visualización de resultados: matriz de confusión y curva ROC
# El código de esta celda está en src/smishing_peru/graficos.py.
from smishing_peru.graficos import graficar_curva_roc, graficar_matriz_confusion

In [ ]:
y_proba_mejor = mejor_modelo.predict_proba(ctx["X_test_txt"])[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
graficar_matriz_confusion(ctx["y_test"], y_pred_final, ctx["label_encoder"], ax=axes[0],
                           titulo=f"Matriz de Confusión - {mejor_nombre}")
graficar_curva_roc(ctx["y_test"], y_proba_mejor, ax=axes[1], titulo=f"Curva ROC - {mejor_nombre}")
plt.tight_layout()
plt.show()

In [ ]:
# Comparación visual de F1-Score y latencia entre los 5 modelos base (configuración ganadora)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.barplot(data=resultados_ganador, x="F1-Score", y="Modelo", hue="Modelo", palette="Blues_d",
            legend=False, ax=axes[0])
minimo = resultados_ganador["F1-Score"].min()
axes[0].set_xlim(max(0.0, minimo - 0.05), 1.0)
axes[0].set_title("F1-Score por modelo")

sns.barplot(data=resultados_ganador, x="Latencia (ms/SMS)", y="Modelo", hue="Modelo", palette="Oranges_d",
            legend=False, ax=axes[1])
axes[1].set_title("Latencia de inferencia por SMS (ms)")

plt.tight_layout()
plt.show()

## 8. Evidencias para la revisión

Se genera un Excel con todas las tablas y un archivo .zip con los datos de cada etapa, el vocabulario TF-IDF y el modelo entrenado.

In [ ]:
# --- 8. Evidencias para la revisión: todo en un Excel y una carpeta comprimida ---
# El Excel junta las tablas que se muestran en la exposición; la carpeta "evidencias/"
# tiene los datos completos de cada etapa (entradas, preprocesado, partición y errores).
import shutil

import sklearn
import xgboost
import sys

# El pipeline completo (preprocesado + TF-IDF + 7 variables + modelo) en un solo objeto.
# Para cargarlo en otro notebook hay que definir antes las celdas 2.2 y 4 (catálogo,
# preprocesar_texto, StylisticFeatureExtractor y PreprocesadorTexto), porque son clases
# y funciones propias. Uso: modelo = joblib.load(...)["pipeline"]; modelo.predict(["texto"]).
joblib.dump({"pipeline": mejor_modelo, "nombre": mejor_nombre, "configuracion": config_ganadora,
             "clases": list(ctx["label_encoder"].classes_),
             "versiones": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__,
                           "xgboost": xgboost.__version__, "numpy": np.__version__, "pandas": pd.__version__}},
            "evidencias/modelo_final.joblib")
if hasattr(mejor_modelo, "named_steps"):
    vectorizador = dict(mejor_modelo.named_steps["variables"].transformer_list)["tfidf"].named_steps["vectorizador"]
    pd.DataFrame({"termino": vectorizador.get_feature_names_out(), "idf": vectorizador.idf_}).sort_values("idf").to_csv(
        "evidencias/07_vocabulario_tfidf.csv", index=False)

tablas_excel = {
    "1_recorrido_datos": tabla_embudo,
    "1b_enlaces_originales": tabla_enlaces,
    "2_composicion_final": tabla_composicion,
    "3_particion": tabla_particion,
    "4_ablaciones": resumen_ablacion,
    "5_modelos": tabla_final,
    "5b_modelos_base": resultados_ganador,
    "6_intervalos_confianza": tabla_ic,
    "7_mcnemar": tabla_mcnemar,
    "8_aporte_generados": tabla_aporte_generados,
    "9_test_por_fuente": tabla_por_fuente,
    "10_similitud_train_test": tabla_similitud,
    "11_lineas_base_atajos": tabla_atajos,
    "11b_telefono": tabla_telefono,
    "11d_particiones": tabla_particiones,
    "11e_gatillo": tabla_gatillo,
    "12_robustez_adversaria": tabla_adversaria,
    "13_curva_aprendizaje": tabla_curva,
    "14_estado_del_arte": tabla_estado_arte,
    "15_replica_datasets": tabla_replica,
    "16_pruebas_a_mano": tabla_pruebas,
}
if tabla_imc25 is not None:
    tablas_excel["11f_imc25"] = tabla_imc25
if tabla_imc25_entrenando is not None:
    tablas_excel["11g_imc25_entrenando"] = tabla_imc25_entrenando

with pd.ExcelWriter("evidencias/resultados_TakedaHiroshi.xlsx") as writer:
    for hoja, tabla in tablas_excel.items():
        con_indice = isinstance(tabla.columns, pd.MultiIndex) or tabla.index.name is not None
        tabla.to_excel(writer, sheet_name=hoja, index=con_indice)

shutil.make_archive("evidencias_TakedaHiroshi", "zip", "evidencias")
print("Contenido de evidencias/:")
for nombre in sorted(os.listdir("evidencias")):
    print(f"  {nombre}  ({os.path.getsize(os.path.join('evidencias', nombre)) / 1024:.0f} KB)")
try:
    from google.colab import files
    files.download("evidencias_TakedaHiroshi.zip")
except ImportError:
    print("\nArchivo listo: evidencias_TakedaHiroshi.zip")

## 9. Experimentos pedidos en la revisión de la exposición

Esta sección responde a los pedidos de la revisión que no cambian el diseño general (el cambio principal, usar los SMS generados solo para entrenar, ya está en la sección 5.0). Usa lo que ya calcularon las secciones 4 y 5 y tarda unos 10 minutos en Colab.

1. **Tabla por pliegue y curvas del mejor modelo (9.1 y 9.2):** cada modelo distinto de la Tabla 7, entrenado en los 5 pliegues, con su F1 de entrenamiento y de validación en cada pliegue. El mejor por validación se aplica al test y se grafican su exactitud y su pérdida para revisar si hay sobreajuste.
2. **El spam no es smishing (9.3):** resultados con y sin los SMISHING que en su dataset original estaban etiquetados como spam.

Las tablas, los gráficos y el modelo se guardan en `experimentos_v7/` y en un .zip (9.4).

### 9.1 Tabla por pliegue: entrenamiento y validación de cada modelo

Cada modelo distinto de la Tabla 7 se entrena 5 veces, una por pliegue de la validación cruzada por grupos (solo con train). En cada pliegue se mide el F1 y la exactitud en los SMS con los que entrenó y en los del pliegue que no vio. Si el F1 de entrenamiento queda muy por encima del de validación, el modelo está memorizando. El mejor por validación, entre los que caben en el celular (la misma regla de la sección 5.3), es el que se aplica al test.

In [ ]:
# --- 9.1 Tabla por pliegue: cada modelo distinto x 5 pliegues de validación cruzada ---
# Es la misma validación cruzada de la sección 5.3 (5 pliegues por grupos dentro de train,
# validando solo con SMS reales), pero mostrando cada pliegue por separado: el F1 y la exactitud en los SMS con los que el modelo
# entrenó (entrenamiento) y en los SMS del pliegue que no vio (validación). El test no se usa
# para elegir: el mejor modelo sale de la media de validación y recién después va al test.
from sklearn.model_selection import cross_validate
from sklearn.base import clone

CARPETA_V7 = "experimentos_v7"
os.makedirs(CARPETA_V7, exist_ok=True)
ctx = ctx_ganador


def _mismos_hiperparametros(a, b, claves=("n_estimators", "max_depth", "min_samples_leaf", "learning_rate", "C")):
    pa, pb = a.get_params(), b.get_params()
    return all(pa.get(f"modelo__{k}") == pb.get(f"modelo__{k}") for k in claves)


# Los 5 modelos base y los 6 afinados o ensambles. Si el Grid Search eligió exactamente los
# hiperparámetros del modelo base, es el mismo modelo y no se entrena dos veces.
modelos_9 = dict(ctx["modelos_entrenados"])
equivalente_base = {"Random Forest afinado (Grid Search)": "Random Forest",
                    "XGBoost afinado (Grid Search)": "XGBoost",
                    "AdaBoost afinado (Grid Search)": "AdaBoost"}
for nombre, modelo in modelos_afinados.items():
    base = equivalente_base.get(nombre)
    if base and _mismos_hiperparametros(modelo, modelos_9[base]):
        print(f"{nombre}: el Grid Search eligió los mismos hiperparámetros que el modelo base; no se repite.")
        continue
    modelos_9[nombre] = modelo

n_pliegues = ctx["cv"].get_n_splits()
print(f"\nEntrenando {len(modelos_9)} modelos x {n_pliegues} pliegues = {len(modelos_9) * n_pliegues} entrenamientos...")
t0 = time.perf_counter()
filas = []
for nombre, modelo in modelos_9.items():
    r = cross_validate(clone(modelo), ctx["X_train_txt"], ctx["y_train"], groups=ctx["grupos_train"],
                       cv=ctx["cv"], scoring=["f1", "accuracy"], return_train_score=True, n_jobs=-1)
    for k in range(n_pliegues):
        filas.append({"Modelo": nombre, "Pliegue": k + 1,
                      "F1 entrenamiento": r["train_f1"][k], "F1 validación": r["test_f1"][k],
                      "Exactitud entrenamiento": r["train_accuracy"][k], "Exactitud validación": r["test_accuracy"][k]})
    print(f"  {nombre}: F1 validación {r['test_f1'].mean():.4f} ± {r['test_f1'].std():.4f}")
print(f"Listo en {time.perf_counter() - t0:.0f} s")
tabla_pliegues = pd.DataFrame(filas)

# Resumen por modelo (la desviación es la poblacional, como en la Tabla 7).
resumen_pliegues = tabla_pliegues.groupby("Modelo", sort=False).agg(
    **{"F1 entrenamiento (media)": ("F1 entrenamiento", "mean"),
       "F1 validación (media)": ("F1 validación", "mean"),
       "F1 validación (desv.)": ("F1 validación", lambda v: v.std(ddof=0)),
       "Exactitud entrenamiento (media)": ("Exactitud entrenamiento", "mean"),
       "Exactitud validación (media)": ("Exactitud validación", "mean")})
resumen_pliegues["Brecha F1 (entrenamiento - validación)"] = (resumen_pliegues["F1 entrenamiento (media)"]
                                                               - resumen_pliegues["F1 validación (media)"])
resumen_pliegues = resumen_pliegues.sort_values("F1 validación (media)", ascending=False)

# Control: la media de validación debe ser la misma "F1 CV" de la tabla de la sección 5.3.
f1_cv_53 = tabla_final.drop_duplicates("Modelo").set_index("Modelo")["F1 CV (5-fold, media)"]
diferencia = (resumen_pliegues["F1 validación (media)"] - f1_cv_53.reindex(resumen_pliegues.index)).abs().max()
print(f"Diferencia máxima con la F1 CV de la sección 5.3: {diferencia:.6f} (debe ser 0)")

# El mejor por validación se aplica al test (entrenado con todo train, igual que en la 5.3),
# con la misma regla de la 5.3: si SOLO_LIGEROS, solo entre los que caben en el celular.
cabe_9 = tabla_final.drop_duplicates("Modelo").set_index("Modelo")["¿Cabe en el celular?"]
resumen_pliegues["¿Cabe en el celular?"] = cabe_9.reindex(resumen_pliegues.index).values
candidatos_9 = resumen_pliegues.index[resumen_pliegues["¿Cabe en el celular?"] == "Sí"] if SOLO_LIGEROS else []
mejor_9 = candidatos_9[0] if len(candidatos_9) else resumen_pliegues.index[0]
y_pred_9 = modelos_9[mejor_9].predict(ctx["X_test_txt"])
tn9, fp9, fn9, tp9 = confusion_matrix(ctx["y_test"], y_pred_9).ravel()
m9 = _metricas(ctx["y_test"], y_pred_9)
print(f"\nMejor modelo por F1 de validación{' (entre los que caben en el celular)' if len(candidatos_9) else ''}: {mejor_9} "
      f"({resumen_pliegues.loc[mejor_9, 'F1 validación (media)']:.4f} ± {resumen_pliegues.loc[mejor_9, 'F1 validación (desv.)']:.4f})"
      f"  |  ¿Es el modelo final de la sección 5.3? {'Sí' if mejor_9 == mejor_nombre else 'No'}")
print(f"En el TEST ({len(ctx['y_test'])} SMS): " + " | ".join(f"{k} {v:.4f}" for k, v in m9.items())
      + f"  |  TN={tn9} FP={fp9} FN={fn9} TP={tp9}")

print("\nF1 de validación de cada modelo en cada pliegue:")
print(tabla_pliegues.pivot(index="Modelo", columns="Pliegue", values="F1 validación")
      .reindex(resumen_pliegues.index).round(4).to_string())
print("\nResumen por modelo (ordenado por F1 de validación):")
resumen_pliegues.round(4)

In [ ]:
# Tabla completa: una fila por modelo y pliegue (F1 y exactitud de entrenamiento y validación).
tabla_pliegues.round(4)

### 9.2 Curvas de exactitud y pérdida del mejor modelo

La exactitud de validación debe subir y la pérdida debe bajar a medida que el modelo ve más SMS. Si al final las curvas de entrenamiento y de validación quedan juntas, no hay sobreajuste.

In [ ]:
# --- 9.2 Curvas del mejor modelo: exactitud (debe subir) y pérdida (debe bajar) ---
# Una SVM no se entrena por épocas como una red neuronal: se resuelve en un solo paso. Por eso
# las curvas se hacen sobre la cantidad de SMS de entrenamiento (10 % ... 100 % de cada pliegue),
# como la curva de aprendizaje de la 5.7 (a), pero ahora con exactitud y con pérdida logarítmica
# (log loss, la misma pérdida que se grafica en una red neuronal, calculada con predict_proba).
#  - Validación: la exactitud sube y la pérdida baja a medida que hay más datos.
#  - Entrenamiento: queda casi en 1, porque el modelo acierta los SMS que ya vio.
#  - Si al final las dos curvas quedan juntas, el modelo no está sobreajustado.
# La curva árbol por árbol (por iteraciones) de XGBoost está en la 5.7 (b).
from sklearn.metrics import log_loss

FOLDS_CURVA_9 = 3
proporciones = np.linspace(0.1, 1.0, 5)
cv_curva = ValidacionSoloReales(n_splits=FOLDS_CURVA_9)  # validación: solo SMS reales
X_tr9, y_tr9, g_tr9 = ctx["X_train_txt"], ctx["y_train"], ctx["grupos_train"]
rng = np.random.RandomState(RANDOM_STATE)

t0 = time.perf_counter()
filas = []
for pliegue, (i_ent, i_val) in enumerate(cv_curva.split(X_tr9, y_tr9, groups=g_tr9), start=1):
    orden = rng.permutation(i_ent)  # los pliegues ya respetan los grupos; aquí solo se toma una porción de train
    for p in proporciones:
        sub = orden[:int(round(p * len(orden)))]
        modelo_p = clone(modelos_9[mejor_9]).fit(X_tr9[sub], y_tr9[sub])
        for parte, idx in [("Entrenamiento", sub), ("Validación", i_val)]:
            filas.append({"Pliegue": pliegue, "Proporción de train": p, "SMS de entrenamiento": len(sub), "Parte": parte,
                          "Exactitud": accuracy_score(y_tr9[idx], modelo_p.predict(X_tr9[idx])),
                          "Pérdida (log loss)": log_loss(y_tr9[idx], modelo_p.predict_proba(X_tr9[idx])[:, 1])})
print(f"Curvas calculadas en {time.perf_counter() - t0:.0f} s "
      f"({FOLDS_CURVA_9} pliegues x {len(proporciones)} tamaños = {FOLDS_CURVA_9 * len(proporciones)} entrenamientos)")
curvas_9 = pd.DataFrame(filas)
tabla_curvas_9 = (curvas_9.groupby(["Proporción de train", "Parte"])
                  .agg(**{"SMS de entrenamiento": ("SMS de entrenamiento", "mean"),
                          "Exactitud": ("Exactitud", "mean"), "Exactitud (desv.)": ("Exactitud", "std"),
                          "Pérdida (log loss)": ("Pérdida (log loss)", "mean"),
                          "Pérdida (desv.)": ("Pérdida (log loss)", "std")})
                  .reset_index())
tabla_curvas_9["SMS de entrenamiento"] = tabla_curvas_9["SMS de entrenamiento"].round().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metrica, titulo in [(axes[0], "Exactitud", "Exactitud (accuracy)"), (axes[1], "Pérdida (log loss)", "Pérdida (log loss)")]:
    desv = "Exactitud (desv.)" if metrica == "Exactitud" else "Pérdida (desv.)"
    for parte, color in [("Entrenamiento", "#4C72B0"), ("Validación", "#DD8452")]:
        t = tabla_curvas_9[tabla_curvas_9["Parte"] == parte]
        ax.plot(t["SMS de entrenamiento"], t[metrica], "o-", color=color, label=parte)
        ax.fill_between(t["SMS de entrenamiento"], t[metrica] - t[desv], t[metrica] + t[desv], color=color, alpha=0.15)
    ax.set_xlabel("SMS usados para entrenar")
    ax.set_title(f"{mejor_9}: {titulo}")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
fig.savefig(f"{CARPETA_V7}/9_2_curvas_mejor_modelo.png", dpi=150, bbox_inches="tight")
plt.show()

ent = tabla_curvas_9[tabla_curvas_9["Parte"] == "Entrenamiento"].reset_index(drop=True)
val = tabla_curvas_9[tabla_curvas_9["Parte"] == "Validación"].reset_index(drop=True)
print(f"Validación: exactitud {val['Exactitud'].iloc[0]:.4f} -> {val['Exactitud'].iloc[-1]:.4f} | "
      f"pérdida {val['Pérdida (log loss)'].iloc[0]:.4f} -> {val['Pérdida (log loss)'].iloc[-1]:.4f}")
print(f"Con todo el pliegue: exactitud entrenamiento {ent['Exactitud'].iloc[-1]:.4f} vs validación "
      f"{val['Exactitud'].iloc[-1]:.4f} (brecha {ent['Exactitud'].iloc[-1] - val['Exactitud'].iloc[-1]:.4f}); "
      f"pérdida {ent['Pérdida (log loss)'].iloc[-1]:.4f} vs {val['Pérdida (log loss)'].iloc[-1]:.4f}")
tabla_curvas_9.round(4)

### 9.3 Con y sin spam

En los datasets originales, "spam" es publicidad no deseada y no siempre es fraude. Para ver cuánto pesan esos SMS, se comparan los resultados con ellos (variante A, la de la sección 5) y sin ellos en train ni en test (variante B).

In [ ]:
# --- 9.3 El spam no es smishing: resultados con y sin los SMISHING que venían de spam ---
# Parte de los SMISHING adaptados vienen de SMS que en su dataset original eran "spam" (publicidad
# o premios, no necesariamente fraude). Variante A: todo el dataset (resultados de la sección 5).
# Variante B: se quitan esos SMS de train y de test, y se reentrenan los 5 modelos base con la
# misma partición y la misma configuración. Los spam son todos SMS adaptados (reales).

from sklearn.model_selection import cross_val_score

X_df, grupos_df = df["text"].values, df["grupo"].values
y_df = ctx["label_encoder"].transform(df["label"])


def comparar_en_particion(idx_ent, idx_prueba, seed=RANDOM_STATE):
    """Entrena los 5 modelos base (con la configuración ganadora) con idx_ent, los compara por
    F1 de validación cruzada (por grupos y validando solo con SMS reales) y los evalúa en idx_prueba."""
    X_ent, y_ent, g_ent = X_df[idx_ent], y_df[idx_ent], grupos_df[idx_ent]
    cv5 = ValidacionSoloReales(n_splits=5, seed=seed)
    filas, modelos = [], {}
    for nombre, clasificador in modelos_base(seed).items():
        pipe = crear_pipeline(clasificador, ctx["usar_logica_difusa"], ctx["quitar_stopwords"])
        cv_f1 = cross_val_score(pipe, X_ent, y_ent, groups=g_ent, cv=cv5, scoring="f1", n_jobs=-1)
        pipe.fit(X_ent, y_ent)
        filas.append({"Modelo": nombre, "F1 CV (media)": cv_f1.mean(), "F1 CV (desv.)": cv_f1.std(),
                      **_metricas(y_df[idx_prueba], pipe.predict(X_df[idx_prueba]))})
        modelos[nombre] = pipe
    tabla = pd.DataFrame(filas).sort_values("F1 CV (media)", ascending=False).reset_index(drop=True)
    return tabla, modelos


etiqueta_original = referencia.drop_duplicates("text").set_index("text")["label_original"]
es_spam = (df["text_original"].map(etiqueta_original) == "spam").values
print("SMS del dataset final según la etiqueta de su dataset original:")
print(pd.crosstab(df["label"], df["text_original"].map(etiqueta_original).fillna("(generado con Gemini)")).to_string())
en_test = np.isin(np.arange(len(df)), ctx["idx_test"])
print(f"\nSMISHING que venían de spam: {es_spam.sum()} ({(es_spam & ~en_test).sum()} en train y {(es_spam & en_test).sum()} en test)")

# (a) El modelo final actual, separado: ¿detecta igual los que venían de spam que los demás?
spam_test = es_spam[ctx["idx_test"]]
smishing_test = ctx["y_test"] == 1
for nombre, m in [("SMISHING que venían de spam", smishing_test & spam_test),
                  ("Los demás SMISHING", smishing_test & ~spam_test)]:
    print(f"  Recall del modelo final en {nombre}: {recall_score(ctx['y_test'][m], y_pred_final[m]):.4f} (n={m.sum()})")

# (b) Variante B: sin spam en train ni en test.
idx_train_b = ctx["idx_train"][~es_spam[ctx["idx_train"]]]
idx_test_b = ctx["idx_test"][~spam_test]
t0 = time.perf_counter()
tabla_sin_spam, modelos_sin_spam = comparar_en_particion(idx_train_b, idx_test_b)
print(f"\nVariante B entrenada en {time.perf_counter() - t0:.0f} s (train {len(idx_train_b)} SMS, test {len(idx_test_b)} SMS)")

columnas = ["F1 CV (media)", "F1-Score", "Precision", "Recall"]
con_spam = resultados_ganador.set_index("Modelo").rename(columns={"F1 CV (5-fold, media)": "F1 CV (media)"})[columnas]
sin_spam = tabla_sin_spam.set_index("Modelo")[columnas]
tabla_spam = pd.concat({"A: con spam": con_spam, "B: sin spam": sin_spam}, axis=1).reindex(sin_spam.index)
tabla_spam[("Diferencia B - A", "F1 test")] = tabla_spam[("B: sin spam", "F1-Score")] - tabla_spam[("A: con spam", "F1-Score")]
print(f"Mejor por F1 CV: con spam {resultados_ganador.loc[0, 'Modelo']} | sin spam {tabla_sin_spam.loc[0, 'Modelo']}")
tabla_spam.round(4)

### 9.4 Guardar los experimentos

Se guardan las tablas de 9.1 a 9.3 (CSV y Excel), el gráfico de 9.2 y el mejor modelo de 9.1 en `experimentos_v7/`, y se descarga `experimentos_v7_TakedaHiroshi.zip`.

In [ ]:
# --- 9.4 Guardar los experimentos: tablas (CSV y Excel), gráfico y el mejor modelo ---
import shutil

tablas_v7 = {
    "9_1_tabla_por_pliegue": tabla_pliegues,
    "9_1_resumen_por_modelo": resumen_pliegues.reset_index(),
    "9_2_curvas_mejor_modelo": tabla_curvas_9,
    "9_3_con_y_sin_spam": tabla_spam,
}
for nombre, tabla in tablas_v7.items():
    tabla.to_csv(f"{CARPETA_V7}/{nombre}.csv", index=isinstance(tabla.columns, pd.MultiIndex))
with pd.ExcelWriter(f"{CARPETA_V7}/experimentos_v7_TakedaHiroshi.xlsx") as writer:
    for nombre, tabla in tablas_v7.items():
        tabla.to_excel(writer, sheet_name=nombre[:31], index=isinstance(tabla.columns, pd.MultiIndex))

# El mejor modelo de la 9.1 (entrenado con todo train), con el mismo formato que la sección 8.
joblib.dump({"pipeline": modelos_9[mejor_9], "nombre": mejor_9, "configuracion": config_ganadora,
             "clases": list(ctx["label_encoder"].classes_),
             "versiones": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__,
                           "xgboost": xgboost.__version__, "numpy": np.__version__, "pandas": pd.__version__}},
            f"{CARPETA_V7}/mejor_modelo_9_1.joblib")

shutil.make_archive("experimentos_v7_TakedaHiroshi", "zip", CARPETA_V7)
print(f"Contenido de {CARPETA_V7}/:")
for nombre in sorted(os.listdir(CARPETA_V7)):
    print(f"  {nombre}  ({os.path.getsize(os.path.join(CARPETA_V7, nombre)) / 1024:.0f} KB)")
try:
    from google.colab import files
    files.download("experimentos_v7_TakedaHiroshi.zip")
except ImportError:
    print("\nArchivo listo: experimentos_v7_TakedaHiroshi.zip")